# 기능 요구사항 전체 → GOLD DATASET v10.4 + 잔존 중복 평가 개선 v10.5

실행할 때마다 해당 문서의 기존 산출물을 삭제하고 **항상 TASK1부터 신규 실행**합니다. 이전 체크포인트는 재사용하지 않습니다.

핵심 흐름: `TASK1 원자 분해 → TASK2 FUR 내부 병합·정규화 및 계보 자동 복구 → 임베딩 후보 검색 → TASK3 전역 중복 판정 → 최종 GOLD → 다중 지표 자동 평가`

TASK3 후보 검색은 기존처럼 재현율 중심으로 넓게 유지합니다. 최종 GOLD의 잔존 중복 평가는 `수행행위 일치 + 요구사항명 핵심어 중복 + 높은 의미 유사도`를 함께 적용해 같은 도메인의 서로 다른 기능이 중복으로 과다 판정되는 문제를 줄였습니다.

평가는 입력 대비 수량 변화, 구조·필드, TASK2/atomic/FUR 계보, 의미 보존, 강한 잔존 중복, 검토 후보, 과병합 위험, 문장 품질을 검사합니다. 정답 GOLD 파일이 있으면 정확 이름과 임베딩 기반 일대일 의미 매칭도 추가합니다.


In [1]:
# 최초 1회 설치
%pip install -q -U \
  "transformers>=4.57.0,<5" \
  "peft>=0.17.1,<1" \
  "accelerate>=1.8,<2" \
  "datasets>=3.0,<5" \
  "huggingface_hub>=0.34,<2" \
  "hf-xet>=1.1,<2" \
  "sentence-transformers>=5.0,<6" \
  "json-repair>=0.44,<1" \
  "python-dotenv>=1.0,<2" \
  "pandas>=2.2,<3" \
  "safetensors>=0.5,<1" \
  "bitsandbytes>=0.46,<1"


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ.pop('HF_HUB_ENABLE_HF_TRANSFER', None)
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
os.environ.setdefault('HF_XET_HIGH_PERFORMANCE', '1')
import errno
import gc
import hashlib
import json
import math
import re
import shutil
import time
import traceback
import uuid
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from dotenv import load_dotenv
from huggingface_hub import hf_hub_download, login
from json_repair import repair_json
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
load_dotenv('/workspace/env', override=True)
HF_TOKEN = os.getenv('HF_TOKEN')
if not HF_TOKEN:
    raise RuntimeError('/workspace/env에서 HF_TOKEN을 찾지 못했습니다.')
login(token=HF_TOKEN, add_to_git_credential=False)
print('Hugging Face 로그인 완료')


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face 로그인 완료


In [3]:
PIPELINE_VERSION = 'v10.4'
HF_DATASET_REPO = 'jaehoony/requirements-4task-dataset'
BASE_MODEL = 'Qwen/Qwen3-VL-8B-Instruct'
STAGE1_ADAPTER_REPO = 'jaehoony/req-qwen3vl-stage1-core'
STAGE3_ADAPTER_REPO = 'jaehoony/req-qwen3vl-stage3-task3doc'
TASK1 = 'TASK1_FUR_ATOMIC_DECOMPOSITION'
TASK2 = 'TASK2_FUR_LOCAL_MERGE_NORMALIZE'
TASK3 = 'TASK3_DOCUMENT_GLOBAL_DEDUP_FINALIZE'
USER_TEST_INPUT_DIR = Path('/workspace/test_input')
OUT_DIR = Path('/workspace/e2e_outputs_requirement_gold_v10_4')
USER_TEST_INPUT_DIR.mkdir(parents=True, exist_ok=True)
USER_INPUT_GLOB = '*_기능전체_입력.json'
EXPECTED_TEST_DOCUMENT_COUNT = 3
RESET_OUTPUT_DIR_BEFORE_RUN = True
START_MODE = 'TASK1_FRESH'
TASK3_COVERAGE_FALLBACK_ENABLED = True
APPLY_GLOBAL_SCOPE_TO_SINGLETONS = True
CONTINUE_ON_DOCUMENT_ERROR = False
RUN_USER_DOCUMENT_TEST = True
SCORE_IF_GOLD_EXISTS = True
JSON_WRITE_RETRY_COUNT = 5
JSON_WRITE_RETRY_SECONDS = 1.0
ATTN_IMPLEMENTATION = 'sdpa'
USE_4BIT = False
MODEL_CONTEXT_LIMIT_FALLBACK = 262144
GENERATION_SAFETY_MARGIN = 2048
MINIMUM_FREE_GPU_GB = 8.0
GENERATION_POLICY = {TASK1: {'multiplier': 1.5, 'minimum': 4096, 'maximum': 8192, 'retry_growth': 1.5, 'max_attempts': 2}, TASK2: {'multiplier': 1.5, 'minimum': 6144, 'maximum': 12288, 'retry_growth': 1.5, 'max_attempts': 2}, TASK3: {'multiplier': 1.5, 'minimum': 2048, 'maximum': 12288, 'retry_growth': 1.5, 'max_attempts': 2}}
EMBEDDING_MODEL_NAME = 'intfloat/multilingual-e5-base'
EMBEDDING_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
EMBEDDING_BATCH_SIZE = 64 if EMBEDDING_DEVICE == 'cuda' else 16
HIGH_SIM_THRESHOLD = 0.92
REVIEW_SIM_THRESHOLD = 0.86
ADAPTIVE_FLOOR_MAX = 0.91
NAME_JACCARD_THRESHOLD = 0.12
LEXICAL_COSINE_THRESHOLD = 0.82
LEXICAL_NAME_JACCARD_THRESHOLD = 0.4
PAIR_SIMILARITY_QUANTILE = 0.97
MUTUAL_TOP_K = 5
GROUP_MIN_PAIR_SIMILARITY = 0.84
TASK3_MAX_GROUP_SIZE = 8
TASK3_MAX_LOCAL_ROUNDS = 3
SCOPE_TOP_K = 3
SCOPE_SIM_THRESHOLD = 0.78
print('=' * 72)
print('[기능전체 → GOLD v10.4 TASK1 신규 시작 + 생성물 종합평가 + TASK3 관계메타 자동복구 설정]')
print('Pipeline version       :', PIPELINE_VERSION)
print('Input dir              :', USER_TEST_INPUT_DIR)
print('Input glob             :', USER_INPUT_GLOB)
print('Expected documents     :', EXPECTED_TEST_DOCUMENT_COUNT)
print('Output dir             :', OUT_DIR)
print('Reset output           :', RESET_OUTPUT_DIR_BEFORE_RUN)
print('Start mode             :', START_MODE)
print('Coverage fallback      :', TASK3_COVERAGE_FALLBACK_ENABLED)
print('Singleton global scope :', APPLY_GLOBAL_SCOPE_TO_SINGLETONS)
print('Embedding model/device :', EMBEDDING_MODEL_NAME, EMBEDDING_DEVICE)
print('Embedding batch size   :', EMBEDDING_BATCH_SIZE)
print('Similarity thresholds :', HIGH_SIM_THRESHOLD, REVIEW_SIM_THRESHOLD)
print('Adaptive floor max     :', ADAPTIVE_FLOOR_MAX)
print('Mutual top-k           :', MUTUAL_TOP_K)
print('Group min pair sim     :', GROUP_MIN_PAIR_SIMILARITY)
print('TASK3 max group size   :', TASK3_MAX_GROUP_SIZE)
print('JSON write retries     :', JSON_WRITE_RETRY_COUNT)
print('Use 4bit               :', USE_4BIT)
print('=' * 72)


[기능전체 → GOLD v10.4 TASK1 신규 시작 + 생성물 종합평가 + TASK3 관계메타 자동복구 설정]
Pipeline version       : v10.4
Input dir              : /workspace/test_input
Input glob             : *_기능전체_입력.json
Expected documents     : 3
Output dir             : /workspace/e2e_outputs_requirement_gold_v10_4
Reset output           : True
Start mode             : TASK1_FRESH
Coverage fallback      : True
Singleton global scope : True
Embedding model/device : intfloat/multilingual-e5-base cuda
Embedding batch size   : 64
Similarity thresholds : 0.92 0.86
Adaptive floor max     : 0.91
Mutual top-k           : 5
Group min pair sim     : 0.84
TASK3 max group size   : 8
JSON write retries     : 5
Use 4bit               : False


In [4]:
from peft import PeftModel
from transformers import AutoConfig, AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration
processor = AutoProcessor.from_pretrained(BASE_MODEL, token=HF_TOKEN)
tokenizer = processor.tokenizer
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
base_config = AutoConfig.from_pretrained(BASE_MODEL, token=HF_TOKEN)
text_config = getattr(base_config, 'text_config', base_config)
MODEL_CONTEXT_LIMIT = int(getattr(text_config, 'max_position_embeddings', MODEL_CONTEXT_LIMIT_FALLBACK))
model_kwargs = {'token': HF_TOKEN, 'device_map': {'': 0}, 'attn_implementation': ATTN_IMPLEMENTATION, 'low_cpu_mem_usage': True}
if USE_4BIT:
    model_kwargs['quantization_config'] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
else:
    model_kwargs['torch_dtype'] = torch.bfloat16
base_model = Qwen3VLForConditionalGeneration.from_pretrained(BASE_MODEL, **model_kwargs)
model = PeftModel.from_pretrained(base_model, STAGE1_ADAPTER_REPO, adapter_name='stage1', token=HF_TOKEN, is_trainable=False, low_cpu_mem_usage=True, autocast_adapter_dtype=False)
model.eval()
torch.backends.cuda.matmul.allow_tf32 = True
model.generation_config.do_sample = False
model.generation_config.temperature = None
model.generation_config.top_p = None
model.generation_config.top_k = None
model.generation_config.pad_token_id = tokenizer.pad_token_id
model.generation_config.eos_token_id = tokenizer.eos_token_id

def ensure_stage3_adapter() -> None:
    if 'stage3' not in model.peft_config:
        print('[stage3 adapter lazy load]')
        model.load_adapter(STAGE3_ADAPTER_REPO, adapter_name='stage3', token=HF_TOKEN, is_trainable=False, low_cpu_mem_usage=True, autocast_adapter_dtype=False)
    model.set_adapter('stage3')
print('Model class    :', type(model).__name__)
print('Initial adapter:', list(model.peft_config.keys()))
print('Context limit  :', f'{MODEL_CONTEXT_LIMIT:,}')
print('GPU            :', torch.cuda.get_device_name(0))
free_b, total_b = torch.cuda.mem_get_info()
print('GPU free/total :', f'{free_b / 1024 ** 3:.2f}GB / {total_b / 1024 ** 3:.2f}GB')


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Model class    : PeftModelForCausalLM
Initial adapter: ['stage1']
Context limit  : 262,144
GPU            : NVIDIA A100-SXM4-80GB
GPU free/total : 44.11GB / 79.25GB


In [5]:
TASK_SEARCH_FOLDERS = {TASK1: ['stage1_core'], TASK2: ['stage1_core'], TASK3: ['stage3_task3_doc', 'stage1_core']}
TASK_ARRAY_KEYS = {TASK1: 'atomic_requirements', TASK2: 'normalized_requirements', TASK3: 'final_requirements'}

def message_text(message: dict) -> str:
    content = message.get('content', '')
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return ''.join((str(item.get('text', '')) for item in content if isinstance(item, dict) and item.get('type') == 'text'))
    return str(content)

def normalize_messages(messages: list[dict], require_assistant: bool=False) -> list[dict]:
    normalized = []
    has_assistant = False
    for message in messages:
        role = message.get('role')
        content = message.get('content')
        if role not in {'system', 'user', 'assistant'}:
            raise ValueError(f'지원하지 않는 role: {role}')
        if role == 'assistant':
            has_assistant = True
        if isinstance(content, str):
            content = [{'type': 'text', 'text': content}]
        elif isinstance(content, list):
            clean = []
            for item in content:
                if not isinstance(item, dict) or item.get('type') != 'text':
                    raise ValueError('이 파이프라인은 text content만 지원합니다.')
                clean.append({'type': 'text', 'text': str(item.get('text', ''))})
            content = clean
        else:
            raise TypeError('message.content 형식 오류')
        normalized.append({'role': role, 'content': content})
    if require_assistant and (not has_assistant):
        raise ValueError('assistant 메시지가 없습니다.')
    return normalized

def load_split(folder: str, split: str):
    filename = f'{folder}/{split}-00000-of-00001.parquet'
    local_path = hf_hub_download(repo_id=HF_DATASET_REPO, repo_type='dataset', filename=filename, token=HF_TOKEN)
    return load_dataset('parquet', data_files={split: local_path}, split=split)

def find_training_contract(task_type: str) -> dict:
    array_key = TASK_ARRAY_KEYS[task_type]
    for folder in TASK_SEARCH_FOLDERS[task_type]:
        try:
            dataset = load_split(folder, 'train')
        except Exception:
            continue
        for row_index, row in enumerate(dataset):
            messages = row.get('messages', [])
            system_messages = [message for message in messages if message.get('role') == 'system']
            user_messages = [message for message in messages if message.get('role') == 'user']
            assistant_messages = [message for message in messages if message.get('role') == 'assistant']
            if not system_messages or not user_messages or (not assistant_messages):
                continue
            try:
                user_obj = json.loads(message_text(user_messages[0]))
                assistant_obj = json.loads(message_text(assistant_messages[0]))
            except Exception:
                continue
            if user_obj.get('task_type') != task_type:
                continue
            if not isinstance(assistant_obj.get(array_key), list):
                continue
            return {'folder': folder, 'row_index': row_index, 'system_prompt': message_text(system_messages[0]), 'user_text': message_text(user_messages[0])}
    raise RuntimeError(f'{task_type} 학습 프롬프트를 찾지 못했습니다.')
TRAINING_CONTRACTS = {task_type: find_training_contract(task_type) for task_type in (TASK1, TASK2, TASK3)}
SYSTEM_PROMPTS = {task_type: contract['system_prompt'] for task_type, contract in TRAINING_CONTRACTS.items()}
for task_type, contract in TRAINING_CONTRACTS.items():
    prompt_hash = hashlib.sha256(contract['system_prompt'].encode('utf-8')).hexdigest()[:12]
    print('[학습 프롬프트]', task_type, 'folder=', contract['folder'], 'row=', contract['row_index'], 'sha256=', prompt_hash)
OUTPUT_SCHEMAS = {TASK1: {'array_key': 'atomic_requirements', 'count_key': 'decomposition_count', 'item_keys': {'atomic_id', 'action_type', 'output_name', 'source_text'}}, TASK2: {'array_key': 'normalized_requirements', 'count_key': 'normalized_requirement_count', 'item_keys': {'task2_id', 'merge_decision', 'merged_from', 'reference_context_ids', 'action_type', 'requirement_name', 'requirement_detail', 'source_requirement_ids'}}, TASK3: {'array_key': 'final_requirements', 'count_key': 'final_requirement_count', 'item_keys': {'gold_id', 'action_type', 'requirement_name', 'requirement_detail', 'source_task2_ids', 'source_atomic_ids', 'sources', 'processing_type', 'merge_basis'}}}


[학습 프롬프트] TASK1_FUR_ATOMIC_DECOMPOSITION folder= stage1_core row= 9 sha256= 15859e872adf
[학습 프롬프트] TASK2_FUR_LOCAL_MERGE_NORMALIZE folder= stage1_core row= 17 sha256= 710ae170bf4f
[학습 프롬프트] TASK3_DOCUMENT_GLOBAL_DEDUP_FINALIZE folder= stage3_task3_doc row= 0 sha256= 67ff972b2950


In [6]:
def dump_json(path: Path, obj: Any) -> Path:
    """
    JSON을 임시 파일에 기록한 뒤 원자적으로 교체합니다.
    Errno 5 같은 일시적인 저장장치 오류는 재시도합니다.
    최종 저장 경로를 반환합니다.
    """
    path = Path(path)
    try:
        serialized = json.dumps(obj, ensure_ascii=False, indent=2)
    except Exception as exc:
        raise RuntimeError(f'JSON 직렬화 실패: {path}') from exc
    path.parent.mkdir(parents=True, exist_ok=True)
    last_error = None
    for attempt in range(1, JSON_WRITE_RETRY_COUNT + 1):
        temp_path = path.with_name(f'.{path.name}.{os.getpid()}.{uuid.uuid4().hex}.tmp')
        try:
            with temp_path.open('w', encoding='utf-8') as file:
                file.write(serialized)
                file.flush()
                os.fsync(file.fileno())
            os.replace(temp_path, path)
            if not path.exists():
                raise OSError(errno.EIO, '저장 후 파일이 존재하지 않음')
            actual_size = path.stat().st_size
            if actual_size <= 0:
                raise OSError(errno.EIO, '저장된 파일 크기가 0임')
            print(f'[JSON 저장 완료] {path} ({actual_size / 1024 ** 2:.2f}MB)', flush=True)
            return path
        except OSError as exc:
            last_error = exc
            try:
                if temp_path.exists():
                    temp_path.unlink()
            except Exception:
                pass
            retryable_errno = {errno.EIO, errno.ENOSPC, errno.ESTALE, errno.EBUSY}
            print(f"[JSON 저장 실패] attempt={attempt}/{JSON_WRITE_RETRY_COUNT}, path={path}, errno={getattr(exc, 'errno', None)}, error={exc}", flush=True)
            if attempt >= JSON_WRITE_RETRY_COUNT or exc.errno not in retryable_errno:
                break
            time.sleep(JSON_WRITE_RETRY_SECONDS * attempt)
    raise OSError(getattr(last_error, 'errno', errno.EIO), f'JSON 저장에 반복 실패했습니다.\n- path: {path}\n- attempts: {JSON_WRITE_RETRY_COUNT}\n- last error: {last_error}')

def load_json(path: Path) -> Any:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    return json.loads(path.read_text(encoding='utf-8'))

def dedupe_preserve(values) -> list[str]:
    if values is None:
        return []
    if isinstance(values, str):
        values = [values]
    seen = set()
    result = []
    for value in values:
        value = str(value).strip()
        if value and value not in seen:
            seen.add(value)
            result.append(value)
    return result
RELATION_REQUIRED_KEYS = {'relation_type', 'processing_result', 'comparison_ids', 'excluded_or_absorbed_ids', 'gold_id', 'preserved_conditions', 'rationale'}
ALLOWED_RELATION_TYPES = {'EXACT_DUPLICATE', 'SEMANTIC_DUPLICATE', 'PARTIAL_OVERLAP', 'PARENT_CHILD', 'RELATED_DISTINCT', 'UNIQUE'}
ALLOWED_PROCESSING_RESULTS = {'MERGED', 'ABSORBED', 'KEPT', 'SEPARATED'}

def _first_nonempty(mapping: dict, keys: tuple[str, ...]) -> str:
    for key in keys:
        value = mapping.get(key)
        if value is not None and str(value).strip():
            return str(value).strip()
    return ''

def _normalize_relation_type(value: str) -> str:
    token = re.sub(r'[^A-Z0-9]+', '_', str(value or '').upper()).strip('_')
    aliases = {
        'EXACT': 'EXACT_DUPLICATE', 'EXACT_DUP': 'EXACT_DUPLICATE', 'EXACT_DUPLICATE': 'EXACT_DUPLICATE',
        'DUPLICATE': 'SEMANTIC_DUPLICATE', 'SEMANTIC': 'SEMANTIC_DUPLICATE', 'SEMANTIC_DUP': 'SEMANTIC_DUPLICATE', 'SEMANTIC_DUPLICATE': 'SEMANTIC_DUPLICATE',
        'PARTIAL': 'PARTIAL_OVERLAP', 'OVERLAP': 'PARTIAL_OVERLAP', 'PARTIAL_OVERLAP': 'PARTIAL_OVERLAP',
        'PARENT_CHILD': 'PARENT_CHILD', 'PARENTCHILD': 'PARENT_CHILD', 'HIERARCHY': 'PARENT_CHILD', 'ABSORBED': 'PARENT_CHILD',
        'RELATED': 'RELATED_DISTINCT', 'DISTINCT': 'RELATED_DISTINCT', 'RELATED_DISTINCT': 'RELATED_DISTINCT', 'SEPARATED': 'RELATED_DISTINCT',
        'UNIQUE': 'UNIQUE', 'KEEP': 'UNIQUE', 'KEPT': 'UNIQUE',
    }
    return aliases.get(token, token)

def _normalize_processing_result(value: str) -> str:
    token = re.sub(r'[^A-Z0-9]+', '_', str(value or '').upper()).strip('_')
    aliases = {
        'MERGE': 'MERGED', 'MERGED': 'MERGED', 'COMBINED': 'MERGED',
        'ABSORB': 'ABSORBED', 'ABSORBED': 'ABSORBED',
        'KEEP': 'KEPT', 'KEPT': 'KEPT', 'PRESERVED': 'KEPT', 'UNIQUE': 'KEPT',
        'SEPARATE': 'SEPARATED', 'SEPARATED': 'SEPARATED', 'DISTINCT': 'SEPARATED',
    }
    return aliases.get(token, token)

def _infer_relation_gold_id(relation: dict, finals: list[dict]) -> str:
    direct = _first_nonempty(relation, ('gold_id', 'final_gold_id', 'target_gold_id', 'result_gold_id'))
    valid_ids = {str(item.get('gold_id', '')).strip() for item in finals if isinstance(item, dict)}
    if direct in valid_ids:
        return direct
    if len(valid_ids) == 1:
        return next(iter(valid_ids))
    relation_ids = set(dedupe_preserve(relation.get('comparison_ids', [])) + dedupe_preserve(relation.get('excluded_or_absorbed_ids', [])) + dedupe_preserve(relation.get('source_task2_ids', [])))
    scored = []
    for final in finals:
        gold_id = str(final.get('gold_id', '')).strip()
        overlap = len(relation_ids & set(dedupe_preserve(final.get('source_task2_ids', []))))
        if gold_id and overlap:
            scored.append((overlap, gold_id))
    scored.sort(reverse=True)
    if scored and (len(scored) == 1 or scored[0][0] > scored[1][0]):
        return scored[0][1]
    return ''

def _infer_relation_type(processing_result: str, rationale_text: str, comparison_ids: list[str], excluded_ids: list[str]) -> str:
    basis = str(rationale_text or '').lower()
    if processing_result == 'ABSORBED' or excluded_ids or any(word in basis for word in ('상위', '하위', '흡수', '포괄', 'parent', 'child')):
        return 'PARENT_CHILD'
    if any(word in basis for word in ('부분', '일부 중복', '고유 조건', 'partial', 'overlap')):
        return 'PARTIAL_OVERLAP'
    if processing_result == 'MERGED':
        return 'SEMANTIC_DUPLICATE'
    if processing_result == 'SEPARATED':
        return 'RELATED_DISTINCT'
    return 'UNIQUE' if len(comparison_ids) <= 1 else 'RELATED_DISTINCT'

def _default_relation_rationale(processing_result: str) -> str:
    messages = {
        'MERGED': '후보 요구사항의 의미 중복 또는 부분 중복을 통합하고 원문 조건을 보존하였다.',
        'ABSORBED': '상위·하위 또는 포함 관계를 기준으로 상세 요구사항에 흡수하였다.',
        'SEPARATED': '관련성은 있으나 수행행위 또는 검수 결과가 달라 별도 요구사항으로 유지하였다.',
        'KEPT': '독립 요구사항으로 판단하여 기존 내용을 유지하였다.',
    }
    return messages.get(processing_result, '후보 요구사항과 최종 GOLD의 관계를 기준으로 처리하였다.')

def repair_task3_relation_metadata(obj: dict, repairs: list[str]) -> None:
    """기존 TASK3 학습 출력의 축약 relation_decisions를 결정적으로 보완한다.

    최종 GOLD 본문과 source_task2_ids는 변경하지 않고 감사용 관계 메타데이터만 보완한다.
    안전하게 추론할 수 없는 gold_id는 채우지 않아 이후 엄격 검증과 재시도가 동작하게 한다.
    """
    finals = [item for item in obj.get('final_requirements', []) if isinstance(item, dict)]
    final_by_gold = {str(item.get('gold_id', '')).strip(): item for item in finals if str(item.get('gold_id', '')).strip()}
    relations = obj.get('relation_decisions', [])
    if not isinstance(relations, list):
        return
    for index, relation in enumerate(relations):
        if not isinstance(relation, dict):
            continue
        prefix = f'relation_decisions[{index}]'
        if not relation.get('comparison_ids'):
            alias_ids = relation.get('source_task2_ids') or relation.get('candidate_ids') or relation.get('input_ids') or []
            relation['comparison_ids'] = dedupe_preserve(alias_ids)
            if relation['comparison_ids']:
                repairs.append(f'{prefix}.comparison_ids<-alias')
        relation['comparison_ids'] = dedupe_preserve(relation.get('comparison_ids', []))
        if not relation.get('excluded_or_absorbed_ids'):
            alias_ids = relation.get('absorbed_ids') or relation.get('excluded_ids') or []
            relation['excluded_or_absorbed_ids'] = dedupe_preserve(alias_ids)
            if relation['excluded_or_absorbed_ids']:
                repairs.append(f'{prefix}.excluded_or_absorbed_ids<-alias')
        relation['excluded_or_absorbed_ids'] = dedupe_preserve(relation.get('excluded_or_absorbed_ids', []))
        preserved = relation.get('preserved_conditions', relation.get('conditions', []))
        relation['preserved_conditions'] = dedupe_preserve(preserved)
        gold_id = _infer_relation_gold_id(relation, finals)
        if gold_id and str(relation.get('gold_id', '')).strip() != gold_id:
            relation['gold_id'] = gold_id
            repairs.append(f'{prefix}.gold_id->{gold_id}')
        final = final_by_gold.get(str(relation.get('gold_id', '')).strip())
        if not relation['comparison_ids'] and final:
            relation['comparison_ids'] = dedupe_preserve(final.get('source_task2_ids', []))
            repairs.append(f'{prefix}.comparison_ids<-final.source_task2_ids')
        raw_result = _first_nonempty(relation, ('processing_result', 'result', 'decision', 'processing', 'decision_result'))
        if not raw_result and final:
            raw_result = str(final.get('processing_type', '')).strip()
        processing_result = _normalize_processing_result(raw_result)
        if not processing_result:
            processing_result = 'MERGED' if len(relation['comparison_ids']) > 1 else 'KEPT'
        if relation.get('processing_result') != processing_result:
            relation['processing_result'] = processing_result
            repairs.append(f'{prefix}.processing_result->{processing_result}')
        rationale = _first_nonempty(relation, ('rationale', 'reason', 'basis', 'decision_basis', 'merge_basis'))
        if not rationale and final:
            rationale = str(final.get('merge_basis', '')).strip()
        raw_type = _first_nonempty(relation, ('relation_type', 'type', 'relation', 'relation_kind', 'category'))
        relation_type = _normalize_relation_type(raw_type)
        if not relation_type:
            relation_type = _infer_relation_type(processing_result, rationale, relation['comparison_ids'], relation['excluded_or_absorbed_ids'])
        if relation.get('relation_type') != relation_type:
            relation['relation_type'] = relation_type
            repairs.append(f'{prefix}.relation_type->{relation_type}')
        if not rationale:
            rationale = _default_relation_rationale(processing_result)
        if relation.get('rationale') != rationale:
            relation['rationale'] = rationale
            repairs.append(f'{prefix}.rationale<-derived')


def canonical_json_sha256(obj: Any) -> str:
    payload = json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(',', ':')).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()

def text_sha256(text: str) -> str:
    return hashlib.sha256(str(text).encode('utf-8')).hexdigest()

def safe_file_component(value: str) -> str:
    cleaned = re.sub('[^0-9A-Za-z가-힣._-]+', '_', str(value).strip()).strip('._')
    return cleaned or hashlib.sha256(str(value).encode('utf-8')).hexdigest()[:16]

def selected_adapter(task_type: str) -> str:
    if task_type in {TASK1, TASK2}:
        model.set_adapter('stage1')
        return 'stage1'
    if task_type == TASK3:
        ensure_stage3_adapter()
        return 'stage3'
    raise ValueError(f'지원하지 않는 task_type: {task_type}')

def serialize_user_payload(task_type: str, user_obj: dict) -> str:
    example_text = TRAINING_CONTRACTS[task_type]['user_text']
    if '\n' in example_text:
        return json.dumps(user_obj, ensure_ascii=False, indent=2)
    return json.dumps(user_obj, ensure_ascii=False)

def build_prompt_messages(task_type: str, user_obj: dict) -> list[dict]:
    return [{'role': 'system', 'content': SYSTEM_PROMPTS[task_type]}, {'role': 'user', 'content': serialize_user_payload(task_type, user_obj)}]

def tokenize_prompt_messages(prompt_messages: list[dict]):
    normalized = normalize_messages(prompt_messages, require_assistant=False)
    inputs = tokenizer.apply_chat_template(normalized, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors='pt')
    inputs.pop('token_type_ids', None)
    return inputs

def normalize_task_output_shape(task_type: str, candidate: dict) -> tuple[dict, list[str]]:
    """
    모델의 의미 내용은 그대로 두고 단수/복수 키, 개수,
    일부 관계 메타데이터만 결정적으로 보정합니다.
    """
    if not isinstance(candidate, dict):
        return (candidate, [])
    obj = dict(candidate)
    repairs = []
    if task_type == TASK1:
        singular_key = 'atomic_requirement'
        array_key = 'atomic_requirements'
        count_key = 'decomposition_count'
    elif task_type == TASK2:
        singular_key = 'normalized_requirement'
        array_key = 'normalized_requirements'
        count_key = 'normalized_requirement_count'
    elif task_type == TASK3:
        singular_key = 'final_requirement'
        array_key = 'final_requirements'
        count_key = 'final_requirement_count'
    else:
        return (obj, repairs)
    current_items = obj.get(array_key)
    if not isinstance(current_items, list):
        singular_value = obj.get(singular_key)
        if isinstance(singular_value, dict):
            obj[array_key] = [dict(singular_value)]
            repairs.append(f'{singular_key}:dict->{array_key}:list')
        elif isinstance(singular_value, list):
            obj[array_key] = [dict(item) if isinstance(item, dict) else item for item in singular_value]
            repairs.append(f'{singular_key}:list->{array_key}:list')
    if task_type == TASK3:
        relations = obj.get('relation_decisions')
        if relations is None:
            obj['relation_decisions'] = []
            repairs.append('relation_decisions:none->list')
        elif isinstance(relations, dict):
            obj['relation_decisions'] = [dict(relations)]
            repairs.append('relation_decisions:dict->list')
        elif isinstance(relations, list):
            obj['relation_decisions'] = [dict(item) if isinstance(item, dict) else item for item in relations]
        finals = obj.get('final_requirements')
        relations = obj.get('relation_decisions', [])
        if isinstance(finals, list) and len(finals) == 1 and isinstance(finals[0], dict):
            only_gold_id = str(finals[0].get('gold_id', '')).strip()
            if only_gold_id:
                for relation_index, relation in enumerate(relations):
                    if not isinstance(relation, dict):
                        continue
                    relation_gold_id = str(relation.get('gold_id', '')).strip()
                    if not relation_gold_id:
                        relation['gold_id'] = only_gold_id
                        repairs.append(f'relation_decisions[{relation_index}].gold_id->{only_gold_id}')
        for relation_index, relation in enumerate(relations):
            if not isinstance(relation, dict):
                continue
            relation['comparison_ids'] = dedupe_preserve(relation.get('comparison_ids', []))
            relation['excluded_or_absorbed_ids'] = dedupe_preserve(relation.get('excluded_or_absorbed_ids', []))
            preserved = relation.get('preserved_conditions')
            if isinstance(preserved, str):
                preserved = preserved.strip()
                relation['preserved_conditions'] = [preserved] if preserved else []
                repairs.append(f'relation_decisions[{relation_index}].preserved_conditions:str->list')
            elif preserved is None:
                relation['preserved_conditions'] = []
                repairs.append(f'relation_decisions[{relation_index}].preserved_conditions:none->list')
    if task_type == TASK3:
        repair_task3_relation_metadata(obj, repairs)
    items = obj.get(array_key)
    if isinstance(items, list):
        declared_count = obj.get(count_key)
        actual_count = len(items)
        if declared_count != actual_count:
            obj[count_key] = actual_count
            repairs.append(f'{count_key}:{declared_count}->{actual_count}')
    return (obj, repairs)

def extract_complete_task_json(text: str, task_type: str) -> tuple[dict, str]:
    raw = text.strip()
    cleaned = re.sub('^```(?:json)?\\s*', '', raw, flags=re.IGNORECASE)
    cleaned = re.sub('\\s*```$', '', cleaned).strip()
    schema = OUTPUT_SCHEMAS[task_type]
    array_key = schema['array_key']
    count_key = schema['count_key']
    candidates = []
    for candidate_text, mode in ((raw, 'strict'), (cleaned, 'code_fence_removed')):
        try:
            parsed = json.loads(candidate_text)
            if isinstance(parsed, dict):
                candidates.append((parsed, mode))
        except Exception:
            pass
    decoder = json.JSONDecoder()
    for match in re.finditer('\\{', cleaned):
        try:
            parsed, _ = decoder.raw_decode(cleaned[match.start():])
            if isinstance(parsed, dict):
                candidates.append((parsed, 'json_object_candidate'))
        except Exception:
            continue
    try:
        repaired = repair_json(cleaned, return_objects=True)
        if isinstance(repaired, dict):
            candidates.append((repaired, 'json_repair'))
    except Exception:
        pass
    valid = []
    for raw_candidate, mode in candidates:
        candidate, repairs = normalize_task_output_shape(task_type, raw_candidate)
        items = candidate.get(array_key)
        if not isinstance(items, list):
            continue
        score = 50
        if candidate.get('task_type') == task_type:
            score += 100
        if count_key and count_key in candidate:
            score += 10
        if task_type == TASK3 and isinstance(candidate.get('relation_decisions'), list):
            score += 20
        score += min(len(items), 100)
        parse_mode = mode
        if repairs:
            parse_mode = f'{mode}+shape_repair'
        valid.append((score, len(json.dumps(candidate, ensure_ascii=False)), candidate, parse_mode, repairs))
    if not valid:
        key_sets = [sorted(candidate.keys()) for candidate, _ in candidates[:20] if isinstance(candidate, dict)]
        raise ValueError(f'{task_type} 전체 JSON 객체를 찾지 못했습니다.\n- expected array: {array_key}\n- discovered keys: {key_sets}\n- raw prefix: {raw[:1000]}')
    valid.sort(key=lambda row: (row[0], row[1]), reverse=True)
    _, _, selected, parse_mode, selected_repairs = valid[0]
    if selected_repairs:
        print(f'[출력 구조 자동 보정] {task_type}: ' + '; '.join(selected_repairs), flush=True)
    return (selected, parse_mode)

def validate_task_output(task_type: str, obj: dict) -> dict:
    """출력 구조를 보정·검증하고 보정된 객체를 반환합니다."""
    if not isinstance(obj, dict):
        raise TypeError(f'{task_type} 출력이 JSON 객체가 아닙니다.')
    obj, repairs = normalize_task_output_shape(task_type, obj)
    if repairs:
        print(f'[검증 전 구조 자동 보정] {task_type}: ' + '; '.join(repairs), flush=True)
    schema = OUTPUT_SCHEMAS[task_type]
    array_key = schema['array_key']
    actual_task_type = obj.get('task_type')
    if actual_task_type is None:
        if isinstance(obj.get(array_key), list):
            obj['task_type'] = task_type
            print(f'[TASK 타입 자동 보정] -> {task_type}', flush=True)
        else:
            raise ValueError(f'task_type과 핵심 배열이 모두 누락되었습니다. expected={task_type}, array={array_key}, keys={sorted(obj.keys())}')
    elif actual_task_type != task_type:
        raise ValueError(f'TASK 타입 불일치: expected={task_type}, actual={actual_task_type}')
    items = obj.get(array_key)
    if not isinstance(items, list):
        raise TypeError(f'{task_type}.{array_key}는 배열이어야 합니다.')
    if not items:
        raise ValueError(f'{task_type}.{array_key}가 0건입니다.')
    id_key = {TASK1: 'atomic_id', TASK2: 'task2_id', TASK3: 'gold_id'}[task_type]
    seen_ids = set()
    for index, item in enumerate(items):
        if not isinstance(item, dict):
            raise TypeError(f'{task_type}.{array_key}[{index}]가 객체가 아닙니다.')
        missing = schema['item_keys'] - set(item)
        if missing:
            raise ValueError(f'{task_type}.{array_key}[{index}] 필수 필드 누락={sorted(missing)} / actual={sorted(item.keys())}')
        item_id = str(item.get(id_key, '')).strip()
        if not item_id:
            raise ValueError(f'{task_type}.{array_key}[{index}].{id_key}가 비어 있습니다.')
        if item_id in seen_ids:
            raise ValueError(f'{task_type} 중복 {id_key}: {item_id}')
        seen_ids.add(item_id)
        if task_type == TASK1:
            for key in ('action_type', 'output_name', 'source_text'):
                if not str(item.get(key, '')).strip():
                    raise ValueError(f'{item_id}.{key}가 비어 있습니다.')
        elif task_type == TASK2:
            item['merged_from'] = dedupe_preserve(item.get('merged_from', []))
            item['reference_context_ids'] = dedupe_preserve(item.get('reference_context_ids', []))
            item['source_requirement_ids'] = dedupe_preserve(item.get('source_requirement_ids', []))
            if not str(item.get('requirement_name', '')).strip():
                raise ValueError(f'{item_id}.requirement_name이 비어 있습니다.')
            if not str(item.get('requirement_detail', '')).strip():
                raise ValueError(f'{item_id}.requirement_detail이 비어 있습니다.')
        elif task_type == TASK3:
            item['source_task2_ids'] = dedupe_preserve(item.get('source_task2_ids', []))
            item['source_atomic_ids'] = dedupe_preserve(item.get('source_atomic_ids', []))
            item['sources'] = dedupe_preserve(item.get('sources', []))
            if not item['source_task2_ids']:
                raise ValueError(f'{item_id}.source_task2_ids가 비어 있습니다.')
            if not str(item.get('requirement_name', '')).strip():
                raise ValueError(f'{item_id}.requirement_name이 비어 있습니다.')
            if not str(item.get('requirement_detail', '')).strip():
                raise ValueError(f'{item_id}.requirement_detail이 비어 있습니다.')
    count_key = schema['count_key']
    if count_key:
        obj[count_key] = len(items)
    if task_type == TASK3:
        relations = obj.get('relation_decisions')
        if relations is None:
            relations = []
            obj['relation_decisions'] = relations
        elif not isinstance(relations, list):
            raise TypeError('TASK3.relation_decisions는 배열이어야 합니다.')
        final_gold_ids = {str(item['gold_id']).strip() for item in items}
        for relation_index, relation in enumerate(relations):
            if not isinstance(relation, dict):
                raise TypeError(f'TASK3.relation_decisions[{relation_index}]가 객체가 아닙니다.')
            missing = RELATION_REQUIRED_KEYS - set(relation)
            if missing:
                raise ValueError(f'TASK3.relation_decisions[{relation_index}] 필수 필드 누락={sorted(missing)}')
            relation['comparison_ids'] = dedupe_preserve(relation.get('comparison_ids', []))
            relation['excluded_or_absorbed_ids'] = dedupe_preserve(relation.get('excluded_or_absorbed_ids', []))
            relation['preserved_conditions'] = dedupe_preserve(relation.get('preserved_conditions', []))
            relation_type = str(relation.get('relation_type', '')).strip()
            processing_result = str(relation.get('processing_result', '')).strip()
            relation_gold_id = str(relation.get('gold_id', '')).strip()
            if relation_type not in ALLOWED_RELATION_TYPES:
                raise ValueError(f'허용되지 않은 relation_type: {relation_type}')
            if processing_result not in ALLOWED_PROCESSING_RESULTS:
                raise ValueError(f'허용되지 않은 processing_result: {processing_result}')
            if relation_gold_id not in final_gold_ids:
                raise ValueError(f'관계 판정이 존재하지 않는 GOLD를 참조합니다: {relation_gold_id}')
            if not str(relation.get('rationale', '')).strip():
                raise ValueError(f'relation_decisions[{relation_index}].rationale이 비어 있습니다.')
    return obj

def gpu_status() -> dict:
    free_b, total_b = torch.cuda.mem_get_info()
    return {'device': torch.cuda.get_device_name(0), 'free_gb': round(free_b / 1024 ** 3, 2), 'used_gb': round((total_b - free_b) / 1024 ** 3, 2), 'total_gb': round(total_b / 1024 ** 3, 2)}

def calculate_max_new_tokens(task_type: str, prompt_tokens: int, explicit_cap: int | None=None) -> int:
    policy = GENERATION_POLICY[task_type]
    if explicit_cap is None:
        estimated = int(prompt_tokens * float(policy['multiplier']))
    else:
        estimated = int(explicit_cap)
    estimated = max(estimated, int(policy['minimum']))
    estimated = min(estimated, int(policy['maximum']))
    available = MODEL_CONTEXT_LIMIT - prompt_tokens - GENERATION_SAFETY_MARGIN
    actual = min(estimated, int(available))
    if actual < 1:
        raise RuntimeError(f'{task_type}: context limit 초과. prompt={prompt_tokens:,}')
    return int(actual)

@torch.inference_mode()
def generate_from_messages(task_type: str, prompt_messages: list[dict], *, raw_log_path: Path | None=None, user_obj_for_log: dict | None=None) -> tuple[dict, str]:
    adapter_name = selected_adapter(task_type)
    policy = GENERATION_POLICY[task_type]
    max_attempts = int(policy['max_attempts'])
    raw_log_path = Path(raw_log_path) if raw_log_path is not None else None
    if raw_log_path is not None:
        raw_log_path.parent.mkdir(parents=True, exist_ok=True)
    base_messages = list(prompt_messages)
    current_messages = list(base_messages)
    explicit_cap = None
    attempts = []
    last_error = None
    for attempt in range(1, max_attempts + 1):
        raw_text = ''
        hit_token_limit = False
        generated = None
        generated_ids = None
        gc.collect()
        torch.cuda.empty_cache()
        cpu_inputs = tokenize_prompt_messages(current_messages)
        prompt_tokens = int(cpu_inputs['input_ids'].shape[-1])
        max_new_tokens = calculate_max_new_tokens(task_type, prompt_tokens, explicit_cap=explicit_cap)
        status_before = gpu_status()
        if status_before['free_gb'] < MINIMUM_FREE_GPU_GB:
            raise RuntimeError(f"{task_type}: GPU 여유 메모리 부족. free={status_before['free_gb']:.2f}GB")
        inputs = {key: value.to(model.device) for key, value in cpu_inputs.items() if isinstance(value, torch.Tensor)}
        input_length = int(inputs['input_ids'].shape[-1])
        print(f"[생성 시작] {task_type} adapter={adapter_name} attempt={attempt}/{max_attempts} prompt={prompt_tokens:,} max_new={max_new_tokens:,} GPU_free={status_before['free_gb']:.2f}GB", flush=True)
        torch.cuda.synchronize()
        started = time.perf_counter()
        try:
            generated = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True, eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id)
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - started
            generated_ids = generated[0, input_length:]
            generated_count = int(generated_ids.shape[-1])
            raw_text = tokenizer.decode(generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False).strip()
            hit_token_limit = generated_count >= max_new_tokens
            attempt_record = {'attempt': attempt, 'adapter_name': adapter_name, 'prompt_tokens': prompt_tokens, 'max_new_tokens': max_new_tokens, 'generated_tokens': generated_count, 'hit_token_limit': hit_token_limit, 'elapsed_seconds': elapsed, 'raw_text': raw_text}
            if raw_log_path is not None:
                attempt_path = raw_log_path.with_name(f'{raw_log_path.stem}_attempt{attempt}.json')
                dump_json(attempt_path, {'task_type': task_type, 'user': user_obj_for_log, **attempt_record})
            if hit_token_limit:
                raise RuntimeError(f'{task_type}: 생성 토큰 한도 도달. 출력 일부 누락 가능성이 있어 재생성합니다.')
            obj, parse_mode = extract_complete_task_json(raw_text, task_type)
            if parse_mode.startswith('json_repair') and (not raw_text.rstrip().endswith('}')):
                raise RuntimeError(f'{task_type}: 불완전 json_repair 결과를 거부합니다.')
            if parse_mode.startswith('json_repair'):
                print(f'[주의] {task_type}: json_repair 사용', flush=True)
            obj = validate_task_output(task_type, obj)
            attempt_record['parse_mode'] = parse_mode
            attempt_record['prediction'] = obj
            attempts.append(attempt_record)
            if raw_log_path is not None:
                dump_json(raw_log_path, {'status': 'SUCCESS', 'task_type': task_type, 'adapter_name': adapter_name, 'user': user_obj_for_log, 'attempts': attempts, 'prediction': obj})
            print(f'[생성 완료] {task_type} generated={generated_count:,} elapsed={elapsed / 60:.2f}분 parse={parse_mode}', flush=True)
            del generated, generated_ids, inputs, cpu_inputs
            gc.collect()
            torch.cuda.empty_cache()
            return (obj, raw_text)
        except Exception as exc:
            last_error = exc
            attempts.append({'attempt': attempt, 'error_type': type(exc).__name__, 'error_message': str(exc), 'traceback': traceback.format_exc(), 'hit_token_limit': hit_token_limit})
            if generated is not None:
                del generated
            if generated_ids is not None:
                del generated_ids
            del inputs, cpu_inputs
            gc.collect()
            torch.cuda.empty_cache()
            if attempt < max_attempts:
                previous_cap = max_new_tokens
                explicit_cap = min(int(previous_cap * float(policy['retry_growth'])), int(policy['maximum']))
                current_messages = base_messages + [{'role': 'user', 'content': f'직전 출력은 완전한 JSON 검증에 실패했다. 설명과 코드펜스 없이 학습된 출력 스키마의 완전한 JSON 객체 하나만 처음부터 다시 출력하라. 검증 오류: {str(exc)[:1500]}'}]
                print(f'[자동 재시도] {task_type}: {type(exc).__name__}', flush=True)
                continue
            if raw_log_path is not None:
                dump_json(raw_log_path.with_name(raw_log_path.stem + '_FAILED.json'), {'status': 'FAILED', 'task_type': task_type, 'adapter_name': adapter_name, 'user': user_obj_for_log, 'attempts': attempts})
            raise
    raise RuntimeError(f'{task_type}: 재시도 후 실패') from last_error

def run_task(task_type: str, user_obj: dict, *, raw_log_path: Path | None=None) -> tuple[dict, str]:
    if user_obj.get('task_type') != task_type:
        raise ValueError(f"task_type 불일치: {task_type} != {user_obj.get('task_type')}")
    return generate_from_messages(task_type, build_prompt_messages(task_type, user_obj), raw_log_path=raw_log_path, user_obj_for_log=user_obj)


In [7]:
_task3_singular_sample = {'task_type': TASK3, 'relation_decisions': [{'comparison_ids': ['T2-000011', 'T2-000039'], 'relation_type': 'PARTIAL_OVERLAP', 'processing_result': 'MERGED', 'rationale': '테스트', 'preserved_conditions': '원문 조건 보존', 'excluded_or_absorbed_ids': []}], 'final_requirement': {'gold_id': 'GOLD-001', 'action_type': '관리', 'requirement_name': '테스트 요구사항', 'requirement_detail': '테스트 요구사항을 제공하여야 한다.', 'source_task2_ids': ['T2-000011', 'T2-000039'], 'source_atomic_ids': [], 'sources': [], 'processing_type': 'MERGED', 'merge_basis': '테스트'}}
_normalized_sample, _sample_repairs = normalize_task_output_shape(TASK3, _task3_singular_sample)
_normalized_sample = validate_task_output(TASK3, _normalized_sample)
assert isinstance(_normalized_sample['final_requirements'], list)
assert len(_normalized_sample['final_requirements']) == 1
assert _normalized_sample['final_requirement_count'] == 1
assert _normalized_sample['relation_decisions'][0]['gold_id'] == 'GOLD-001'
assert isinstance(_normalized_sample['relation_decisions'][0]['preserved_conditions'], list)
print('[회귀 테스트 통과] TASK3 final_requirement 단수형 자동 보정')
del (_task3_singular_sample, _normalized_sample, _sample_repairs)

_task3_missing_relation_meta = {
    'task_type': TASK3,
    'final_requirements': [{
        'gold_id': 'GOLD-001', 'action_type': '조회', 'requirement_name': '학생 현황 조회 기능',
        'requirement_detail': '학생 현황을 조회할 수 있어야 한다.', 'source_task2_ids': ['T2-001', 'T2-007'],
        'source_atomic_ids': [], 'sources': ['SFR-001'], 'processing_type': 'MERGED',
        'merge_basis': '두 후보가 동일한 조회 결과를 제공하므로 통합'
    }],
    'relation_decisions': [{
        'comparison_ids': ['T2-001', 'T2-007'], 'excluded_or_absorbed_ids': [],
        'gold_id': 'GOLD-001', 'preserved_conditions': []
    }]
}
_task3_missing_relation_meta = validate_task_output(TASK3, _task3_missing_relation_meta)
_relation = _task3_missing_relation_meta['relation_decisions'][0]
assert _relation['relation_type'] == 'SEMANTIC_DUPLICATE'
assert _relation['processing_result'] == 'MERGED'
assert _relation['rationale']
print('[회귀 테스트 통과] TASK3 relation_decisions 누락 메타데이터 자동 복구')
del _task3_missing_relation_meta, _relation



[회귀 테스트 통과] TASK3 final_requirement 단수형 자동 보정
[검증 전 구조 자동 보정] TASK3_DOCUMENT_GLOBAL_DEDUP_FINALIZE: relation_decisions[0].processing_result->MERGED; relation_decisions[0].relation_type->SEMANTIC_DUPLICATE; relation_decisions[0].rationale<-derived; final_requirement_count:None->1
[회귀 테스트 통과] TASK3 relation_decisions 누락 메타데이터 자동 복구


In [8]:
def normalize_atomic_ids(source_fur_id: str, atomics: list[dict]) -> list[dict]:
    normalized = []
    seen = set()
    for index, raw_item in enumerate(atomics, start=1):
        item = dict(raw_item)
        current_atomic_id = str(item.get('atomic_id', '')).strip() or f'A-{index:03d}'
        original_atomic_id = str(item.get('original_atomic_id', current_atomic_id.split('::')[-1])).strip() or f'A-{index:03d}'
        global_atomic_id = f'{source_fur_id}::{original_atomic_id}'
        if global_atomic_id in seen:
            raise ValueError(f'TASK1 중복 atomic_id: {global_atomic_id}')
        seen.add(global_atomic_id)
        item['original_atomic_id'] = original_atomic_id
        item['atomic_id'] = global_atomic_id
        item['source_fur_id'] = source_fur_id
        normalized.append(item)
    return normalized

def validate_task1_result(source_fur_id: str, atomics: list[dict]) -> None:
    if not atomics:
        raise ValueError(f'{source_fur_id}: TASK1 결과가 0건입니다.')
    seen = set()
    for index, item in enumerate(atomics):
        atomic_id = str(item.get('atomic_id', '')).strip()
        if not atomic_id:
            raise ValueError(f'{source_fur_id}: atomic[{index}] ID 누락')
        if atomic_id in seen:
            raise ValueError(f'{source_fur_id}: 중복 atomic_id={atomic_id}')
        seen.add(atomic_id)
        if item.get('source_fur_id') != source_fur_id:
            raise ValueError(f'{atomic_id}: source_fur_id 불일치')
        for key in ('action_type', 'output_name', 'source_text'):
            if not str(item.get(key, '')).strip():
                raise ValueError(f'{atomic_id}.{key}가 비어 있습니다.')

def canonical_atomic_reference(source_fur_id: str, atomic_id: str) -> str:
    atomic_id = str(atomic_id).strip()
    if not atomic_id:
        return ''
    if '::' in atomic_id:
        return atomic_id
    return f'{source_fur_id}::{atomic_id}'

def normalize_task2_local_output(source_fur_id: str, normalized: list[dict]) -> list[dict]:
    result = []
    for item in normalized:
        item = dict(item)
        item['source_fur_id'] = source_fur_id
        item['source_requirement_ids'] = dedupe_preserve(item.get('source_requirement_ids', []) + [source_fur_id])
        item['merged_from'] = dedupe_preserve((canonical_atomic_reference(source_fur_id, value) for value in item.get('merged_from', [])))
        item['reference_context_ids'] = dedupe_preserve((canonical_atomic_reference(source_fur_id, value) for value in item.get('reference_context_ids', [])))
        result.append(item)
    return result

def _lineage_tokens(value: Any) -> set[str]:
    return {token for token in re.findall(r'[0-9A-Za-z가-힣]+', str(value).lower()) if len(token) >= 2}

def _task2_owner_score(atomic: dict, item: dict, atomic_id: str) -> tuple[float, int]:
    merged_from = dedupe_preserve(item.get('merged_from', []))
    atomic_action = str(atomic.get('action_type', '')).strip()
    item_action = str(item.get('action_type', '')).strip()
    atomic_text = ' '.join((str(atomic.get('output_name', '')), str(atomic.get('source_text', '')), str(atomic.get('requirement_detail', ''))))
    item_text = ' '.join((str(item.get('requirement_name', '')), str(item.get('requirement_detail', ''))))
    left, right = _lineage_tokens(atomic_text), _lineage_tokens(item_text)
    overlap = len(left & right) / max(1, len(left | right))
    score = overlap * 10.0 + (3.0 if atomic_action and atomic_action == item_action else 0.0)
    if merged_from == [atomic_id]:
        score += 100.0  # 이 atomic을 제거하면 주 계보가 사라지는 항목을 우선 보존
    return score, -len(merged_from)

def repair_task2_primary_assignments(atomics: list[dict], normalized: list[dict]) -> tuple[list[dict], list[dict]]:
    atomic_by_id = {str(item['atomic_id']).strip(): item for item in atomics}
    repaired = [dict(item) for item in normalized]
    assignments = defaultdict(list)
    for index, item in enumerate(repaired):
        item['merged_from'] = dedupe_preserve(item.get('merged_from', []))
        item['reference_context_ids'] = dedupe_preserve(item.get('reference_context_ids', []))
        for atomic_id in item['merged_from']:
            assignments[atomic_id].append(index)
    records = []
    for atomic_id, indices in assignments.items():
        if len(indices) <= 1:
            continue
        atomic = atomic_by_id.get(atomic_id, {})
        owner_index = max(indices, key=lambda index: (_task2_owner_score(atomic, repaired[index], atomic_id), -index))
        owner_task2_id = str(repaired[owner_index].get('task2_id', '')).strip()
        moved_to_reference = []
        for index in indices:
            if index == owner_index:
                continue
            item = repaired[index]
            item['merged_from'] = [value for value in item['merged_from'] if value != atomic_id]
            item['reference_context_ids'] = dedupe_preserve(item['reference_context_ids'] + [atomic_id])
            moved_to_reference.append(str(item.get('task2_id', '')).strip())
        records.append({'atomic_id': atomic_id, 'kept_in_merged_from': owner_task2_id, 'moved_to_reference_context_ids': moved_to_reference})
    dropped = []
    kept = []
    for item in repaired:
        if item.get('merged_from'):
            kept.append(item)
        else:
            dropped.append({'task2_id': str(item.get('task2_id', '')).strip(), 'reason': '중복 주 계보 정리 후 merged_from이 비어 과분해 결과로 판단', 'reference_context_ids': item.get('reference_context_ids', [])})
    records.extend({'dropped_unanchored_task2': item} for item in dropped)
    return kept, records

def validate_task2_atomic_coverage(atomics: list[dict], normalized: list[dict]) -> None:
    expected_ids = {str(item['atomic_id']).strip() for item in atomics}
    referenced_ids = set()
    primary_assignment = defaultdict(list)
    for item in normalized:
        task2_id = str(item.get('task2_id', '')).strip()
        merged_from = dedupe_preserve(item.get('merged_from', []))
        reference_ids = dedupe_preserve(item.get('reference_context_ids', []))
        referenced_ids.update(merged_from)
        referenced_ids.update(reference_ids)
        for atomic_id in merged_from:
            primary_assignment[atomic_id].append(task2_id)
    unknown = sorted(referenced_ids - expected_ids)
    missing_reference = sorted(expected_ids - referenced_ids)
    missing_primary = sorted(expected_ids - set(primary_assignment))
    duplicate_primary = {atomic_id: task2_ids for atomic_id, task2_ids in primary_assignment.items() if len(set(task2_ids)) > 1}
    if unknown:
        raise ValueError(f'TASK2가 생성한 알 수 없는 atomic ID: {unknown}')
    if missing_reference:
        raise ValueError(f'TASK2에서 완전히 누락된 atomic ID: {missing_reference}')
    if missing_primary:
        raise ValueError(f'TASK2 merged_from 주 계보가 없는 atomic ID: {missing_primary}')
    if duplicate_primary:
        raise ValueError(f'TASK2 merged_from에 중복 배정된 atomic ID: {duplicate_primary}')

def stage_task1(doc_id: str, doc_name: str, fur: dict, *, raw_log_path: Path) -> list[dict]:
    required = {'requirement_id', 'requirement_name', 'requirement_type', 'requirement_definition', 'requirement_detail'}
    missing = required - set(fur)
    if missing:
        raise ValueError(f'TASK1 입력 필드 누락: {sorted(missing)}')
    user_obj = {'task_type': TASK1, 'document_id': doc_id, 'document_name': doc_name, 'requirement': fur}
    obj, _ = run_task(TASK1, user_obj, raw_log_path=raw_log_path)
    source_fur_id = str(fur['requirement_id']).strip()
    atomics = normalize_atomic_ids(source_fur_id, obj['atomic_requirements'])
    validate_task1_result(source_fur_id, atomics)
    return atomics

def stage_task2(doc_id: str, source_requirement_id: str, atomics: list[dict], *, raw_log_path: Path) -> list[dict]:
    if not isinstance(atomics, list) or not atomics:
        raise TypeError('TASK2 atomic_requirements는 1건 이상의 배열이어야 합니다.')
    validate_task1_result(source_requirement_id, atomics)
    atomic_ids = [str(item['atomic_id']).strip() for item in atomics]
    user_obj = {'task_type': TASK2, 'document_id': doc_id, 'source_requirement_id': source_requirement_id, 'atomic_requirements': atomics, 'reference_contexts': [], 'lineage_constraints': {'all_atomic_ids': atomic_ids, 'merged_from_rule': '각 atomic_id는 normalized_requirements 전체의 merged_from에 정확히 1회만 배정한다.', 'reference_context_rule': '다른 요구사항의 참고 근거로 재사용할 때만 reference_context_ids에 넣는다.', 'no_new_ids': True}}
    obj, _ = run_task(TASK2, user_obj, raw_log_path=raw_log_path)
    normalized = normalize_task2_local_output(source_requirement_id, obj['normalized_requirements'])
    normalized, repair_records = repair_task2_primary_assignments(atomics, normalized)
    if repair_records:
        repair_path = raw_log_path.with_name(f'{raw_log_path.stem}_lineage_repair.json')
        dump_json(repair_path, {'task_type': TASK2, 'document_id': doc_id, 'source_requirement_id': source_requirement_id, 'repair_count': len(repair_records), 'repairs': repair_records, 'normalized_requirements_after_repair': normalized})
        print(f'[TASK2 계보 자동 복구] fur={source_requirement_id}, repairs={len(repair_records)}, log={repair_path}', flush=True)
    validate_task2_atomic_coverage(atomics, normalized)
    return normalized

def normalize_task2_global_ids(candidates: list[dict]) -> list[dict]:
    normalized = []
    original_counts = defaultdict(int)
    for index, raw_item in enumerate(candidates, start=1):
        item = dict(raw_item)
        original_task2_id = str(item.get('task2_id', '')).strip() or f'LOCAL-T2-{index:06d}'
        original_counts[original_task2_id] += 1
        item['original_task2_id'] = original_task2_id
        item['task2_id'] = f'T2-{index:06d}'
        item['source_requirement_ids'] = dedupe_preserve(item.get('source_requirement_ids', []))
        source_fur_id = str(item.get('source_fur_id', '')).strip()
        if not source_fur_id and item['source_requirement_ids']:
            source_fur_id = item['source_requirement_ids'][0]
        if source_fur_id:
            item['source_requirement_ids'] = dedupe_preserve(item['source_requirement_ids'] + [source_fur_id])
        item['source_fur_id'] = source_fur_id
        item['merged_from'] = dedupe_preserve((canonical_atomic_reference(source_fur_id, value) if source_fur_id else str(value).strip() for value in item.get('merged_from', [])))
        item['reference_context_ids'] = dedupe_preserve((canonical_atomic_reference(source_fur_id, value) if source_fur_id else str(value).strip() for value in item.get('reference_context_ids', [])))
        item['action_type'] = str(item.get('action_type', '미지정')).strip() or '미지정'
        item['requirement_name'] = str(item.get('requirement_name', '')).strip()
        item['requirement_detail'] = str(item.get('requirement_detail', '')).strip()
        if not item['requirement_name']:
            raise ValueError(f"{item['task2_id']}.requirement_name이 비어 있습니다.")
        if not item['requirement_detail']:
            raise ValueError(f"{item['task2_id']}.requirement_detail이 비어 있습니다.")
        normalized.append(item)
    duplicate_original_ids = {key: value for key, value in original_counts.items() if value > 1}
    print(f"[TASK2 전역 ID 정규화] count={len(normalized)}, duplicated_local_id_types={len(duplicate_original_ids)}, range={normalized[0]['task2_id']}~{normalized[-1]['task2_id']}", flush=True)
    return normalized

def validate_document_task2_lineage(all_atomics_by_fur: dict[str, list[dict]], candidates: list[dict]) -> None:
    all_atomics = [item for atomics in all_atomics_by_fur.values() for item in atomics]
    validate_task2_atomic_coverage(all_atomics, candidates)

def extract_saved_task2_candidates(saved_obj: Any) -> list[dict]:
    if isinstance(saved_obj, list):
        candidates = saved_obj
    elif isinstance(saved_obj, dict):
        candidates = None
        for key in ('candidate_requirements', 'normalized_requirements', 'task2_candidates', 'requirements'):
            value = saved_obj.get(key)
            if isinstance(value, list):
                candidates = value
                print(f'[TASK2 저장 배열 선택] key={key}, count={len(value)}')
                break
        if candidates is None:
            raise ValueError('저장된 TASK2 후보 배열을 찾지 못했습니다.')
    else:
        raise TypeError('저장된 TASK2 파일의 최상위 형식이 올바르지 않습니다.')
    if not candidates:
        raise ValueError('저장된 TASK2 후보가 0건입니다.')
    return normalize_task2_global_ids(candidates)


In [9]:
TOKEN_PATTERN = re.compile('[가-힣A-Za-z0-9]+')
_embedding_model = None

def get_embedding_model():
    global _embedding_model
    if _embedding_model is None:
        _embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=EMBEDDING_DEVICE)
    return _embedding_model

def text_tokens(text: str) -> set[str]:
    return {token.lower() for token in TOKEN_PATTERN.findall(str(text)) if len(token) >= 2}

def jaccard(text_a: str, text_b: str) -> float:
    tokens_a = text_tokens(text_a)
    tokens_b = text_tokens(text_b)
    if not tokens_a or not tokens_b:
        return 0.0
    return len(tokens_a & tokens_b) / len(tokens_a | tokens_b)

def source_overlap(left: dict, right: dict) -> bool:
    """
    출처 중첩은 분석용 메타데이터로만 사용합니다.
    같은 FUR 출처라는 이유만으로 유사 그룹을 연결하지 않습니다.
    """
    return bool(set(left.get('source_requirement_ids', [])) & set(right.get('source_requirement_ids', [])))

def embedding_text(item: dict) -> str:
    return f"query: 수행행위: {item.get('action_type', '미지정')}\n요구사항명: {item.get('requirement_name', '')}\n상세내용: {item.get('requirement_detail', '')}"

def similarity_distribution(matrix: np.ndarray) -> dict:
    size = int(matrix.shape[0])
    if size < 2:
        return {'pair_count': 0, 'min': None, 'q50': None, 'q75': None, 'q90': None, 'q95': None, 'q97': None, 'q99': None, 'max': None}
    upper = matrix[np.triu_indices(size, k=1)]
    return {'pair_count': int(upper.size), 'min': round(float(np.min(upper)), 6), 'q50': round(float(np.quantile(upper, 0.5)), 6), 'q75': round(float(np.quantile(upper, 0.75)), 6), 'q90': round(float(np.quantile(upper, 0.9)), 6), 'q95': round(float(np.quantile(upper, 0.95)), 6), 'q97': round(float(np.quantile(upper, 0.97)), 6), 'q99': round(float(np.quantile(upper, 0.99)), 6), 'max': round(float(np.max(upper)), 6)}

def mutual_top_k_sets(similarity_matrix: np.ndarray, top_k: int) -> list[set[int]]:
    size = int(similarity_matrix.shape[0])
    result = []
    for index in range(size):
        scores = similarity_matrix[index].astype(float).copy()
        scores[index] = -np.inf
        count = min(max(int(top_k), 1), max(size - 1, 0))
        if count == 0:
            result.append(set())
            continue
        neighbor_indices = np.argsort(-scores)[:count]
        result.append({int(value) for value in neighbor_indices if np.isfinite(scores[int(value)])})
    return result

def all_cross_pairs_above(left_group: set[int], right_group: set[int], similarity_matrix: np.ndarray, minimum: float) -> bool:
    for left in left_group:
        for right in right_group:
            if float(similarity_matrix[left, right]) < minimum:
                return False
    return True

def complete_link_groups(item_count: int, eligible_edges: list[dict], similarity_matrix: np.ndarray, max_group_size: int, minimum_pair_similarity: float) -> tuple[list[list[int]], list[int]]:
    """
    높은 점수의 쌍부터 그룹을 만들되,
    그룹에 새 항목을 추가하거나 두 그룹을 합칠 때
    모든 교차 쌍이 minimum 이상인지 확인합니다.

    단순 연결 전이로 거대 컴포넌트가 만들어지지 않습니다.
    """
    groups: list[set[int]] = []
    item_to_group: dict[int, int] = {}
    sorted_edges = sorted(eligible_edges, key=lambda row: (row['cosine_similarity'], row['name_jaccard']), reverse=True)
    for edge in sorted_edges:
        left = int(edge['left_index'])
        right = int(edge['right_index'])
        left_group_no = item_to_group.get(left)
        right_group_no = item_to_group.get(right)
        if left_group_no is None and right_group_no is None:
            if float(similarity_matrix[left, right]) >= minimum_pair_similarity:
                group_no = len(groups)
                groups.append({left, right})
                item_to_group[left] = group_no
                item_to_group[right] = group_no
            continue
        if left_group_no is not None and left_group_no == right_group_no:
            continue
        if left_group_no is not None and right_group_no is None:
            group = groups[left_group_no]
            if len(group) + 1 <= max_group_size and all_cross_pairs_above(group, {right}, similarity_matrix, minimum_pair_similarity):
                group.add(right)
                item_to_group[right] = left_group_no
            continue
        if left_group_no is None and right_group_no is not None:
            group = groups[right_group_no]
            if len(group) + 1 <= max_group_size and all_cross_pairs_above({left}, group, similarity_matrix, minimum_pair_similarity):
                group.add(left)
                item_to_group[left] = right_group_no
            continue
        if left_group_no is not None and right_group_no is not None and (left_group_no != right_group_no):
            left_group = groups[left_group_no]
            right_group = groups[right_group_no]
            if not left_group or not right_group:
                continue
            if len(left_group) + len(right_group) <= max_group_size and all_cross_pairs_above(left_group, right_group, similarity_matrix, minimum_pair_similarity):
                merged = left_group | right_group
                groups[left_group_no] = merged
                groups[right_group_no] = set()
                for item_index in merged:
                    item_to_group[item_index] = left_group_no
    similar_groups = [sorted(group) for group in groups if len(group) >= 2]
    similar_groups.sort(key=lambda group: min(group))
    grouped_indices = {index for group in similar_groups for index in group}
    singleton_indices = [index for index in range(item_count) if index not in grouped_indices]
    return (similar_groups, singleton_indices)

def build_similarity_groups(doc_id: str, candidates: list[dict], output_dir: Path) -> dict:
    if not candidates:
        raise ValueError('임베딩 후보가 0건입니다.')
    model_for_embedding = get_embedding_model()
    texts = [embedding_text(item) for item in candidates]
    embeddings = model_for_embedding.encode(texts, batch_size=EMBEDDING_BATCH_SIZE, show_progress_bar=True, normalize_embeddings=True, convert_to_numpy=True)
    similarity_matrix = embeddings @ embeddings.T
    distribution = similarity_distribution(similarity_matrix)
    if len(candidates) >= 2:
        upper = similarity_matrix[np.triu_indices(len(candidates), k=1)]
        adaptive_floor = min(max(float(REVIEW_SIM_THRESHOLD), float(np.quantile(upper, PAIR_SIMILARITY_QUANTILE))), float(ADAPTIVE_FLOOR_MAX))
    else:
        adaptive_floor = float(REVIEW_SIM_THRESHOLD)
    top_k_sets = mutual_top_k_sets(similarity_matrix, MUTUAL_TOP_K)
    all_pairs = []
    eligible_edges = []
    for left_index in range(len(candidates)):
        for right_index in range(left_index + 1, len(candidates)):
            left = candidates[left_index]
            right = candidates[right_index]
            cosine = float(similarity_matrix[left_index, right_index])
            same_action = left.get('action_type') == right.get('action_type')
            overlap_source = source_overlap(left, right)
            name_overlap = jaccard(left.get('requirement_name', ''), right.get('requirement_name', ''))
            mutual_neighbor = right_index in top_k_sets[left_index] and left_index in top_k_sets[right_index]
            high_match = cosine >= HIGH_SIM_THRESHOLD
            evidence_match = mutual_neighbor and cosine >= adaptive_floor and (same_action or name_overlap >= NAME_JACCARD_THRESHOLD)
            lexical_match = name_overlap >= LEXICAL_NAME_JACCARD_THRESHOLD and cosine >= LEXICAL_COSINE_THRESHOLD
            action_match = same_action and name_overlap >= NAME_JACCARD_THRESHOLD and (cosine >= REVIEW_SIM_THRESHOLD)
            connected = high_match or evidence_match or lexical_match or action_match
            reasons = []
            if high_match:
                reasons.append('MUTUAL_TOPK_HIGH_COSINE')
            if evidence_match:
                reasons.append('MUTUAL_TOPK_ADAPTIVE_WITH_EVIDENCE')
            if lexical_match:
                reasons.append('LEXICAL_NAME_OVERLAP')
            if action_match:
                reasons.append('ACTION_AND_NAME_EVIDENCE')
            record = {'left_index': left_index, 'right_index': right_index, 'left_id': left['task2_id'], 'right_id': right['task2_id'], 'cosine_similarity': round(cosine, 6), 'same_action_type': same_action, 'source_overlap': overlap_source, 'name_jaccard': round(name_overlap, 6), 'mutual_top_k': mutual_neighbor, 'adaptive_floor': round(adaptive_floor, 6), 'high_match': high_match, 'evidence_match': evidence_match, 'lexical_match': lexical_match, 'action_match': action_match, 'connect': connected, 'reason': reasons}
            all_pairs.append(record)
            if connected:
                eligible_edges.append(record)
    similar_components, singleton_indices = complete_link_groups(item_count=len(candidates), eligible_edges=eligible_edges, similarity_matrix=similarity_matrix, max_group_size=TASK3_MAX_GROUP_SIZE, minimum_pair_similarity=GROUP_MIN_PAIR_SIMILARITY)
    group_rows = []
    for group_no, component in enumerate(similar_components, start=1):
        pair_scores = [float(similarity_matrix[left, right]) for position, left in enumerate(component) for right in component[position + 1:]]
        group_rows.append({'group_no': group_no, 'group_type': 'SIMILAR', 'size': len(component), 'task2_ids': [candidates[index]['task2_id'] for index in component], 'min_pair_similarity': round(min(pair_scores), 6), 'average_pair_similarity': round(float(np.mean(pair_scores)), 6), 'max_pair_similarity': round(max(pair_scores), 6), 'exceeds_group_limit': False})
    for singleton_index in singleton_indices:
        group_rows.append({'group_no': None, 'group_type': 'SINGLETON', 'size': 1, 'task2_ids': [candidates[singleton_index]['task2_id']], 'min_pair_similarity': None, 'average_pair_similarity': None, 'max_pair_similarity': None, 'exceeds_group_limit': False})
    output_dir.mkdir(parents=True, exist_ok=True)
    dump_json(output_dir / '00_similarity_distribution.json', {'document_id': doc_id, 'candidate_count': len(candidates), 'distribution': distribution, 'adaptive_floor': round(adaptive_floor, 6), 'settings': {'high_similarity': HIGH_SIM_THRESHOLD, 'review_similarity': REVIEW_SIM_THRESHOLD, 'pair_quantile': PAIR_SIMILARITY_QUANTILE, 'mutual_top_k': MUTUAL_TOP_K, 'name_jaccard': NAME_JACCARD_THRESHOLD, 'lexical_cosine': LEXICAL_COSINE_THRESHOLD, 'lexical_name_jaccard': LEXICAL_NAME_JACCARD_THRESHOLD, 'adaptive_floor_max': ADAPTIVE_FLOOR_MAX, 'group_min_pair': GROUP_MIN_PAIR_SIMILARITY, 'max_group_size': TASK3_MAX_GROUP_SIZE}})
    dump_json(output_dir / '01_similarity_edges.json', {'document_id': doc_id, 'adaptive_floor': round(adaptive_floor, 6), 'connected_edges': eligible_edges})
    dump_json(output_dir / '02_similarity_groups.json', {'document_id': doc_id, 'candidate_count': len(candidates), 'similar_component_count': len(similar_components), 'similar_component_item_count': sum((len(component) for component in similar_components)), 'singleton_count': len(singleton_indices), 'groups': group_rows})
    pd.DataFrame(eligible_edges).to_csv(output_dir / '01_similarity_edges.csv', index=False, encoding='utf-8-sig')
    pd.DataFrame(group_rows).to_csv(output_dir / '02_similarity_groups.csv', index=False, encoding='utf-8-sig')
    print('=' * 72)
    print('[임베딩 유사 후보 검색 v10]')
    print('전체 TASK2 후보     :', len(candidates))
    print('유사도 분포 q95/q97 :', distribution['q95'], distribution['q97'])
    print('적응형 유사도 하한  :', round(adaptive_floor, 6))
    print('유효 유사 쌍        :', len(eligible_edges))
    print('유사 그룹           :', len(similar_components))
    print('유사 그룹 항목      :', sum((len(component) for component in similar_components)))
    print('독립 항목           :', len(singleton_indices))
    print('최대 그룹 크기      :', max((len(component) for component in similar_components), default=0))
    print('=' * 72)
    if similar_components and max((len(component) for component in similar_components)) > TASK3_MAX_GROUP_SIZE:
        raise RuntimeError('유사 그룹 최대 크기 제한이 적용되지 않았습니다.')
    return {'embeddings': embeddings, 'similarity_matrix': similarity_matrix, 'similar_components': similar_components, 'singleton_indices': singleton_indices, 'group_rows': group_rows, 'connected_edges': eligible_edges, 'similarity_distribution': distribution, 'adaptive_floor': adaptive_floor}

def split_indices_by_similarity(indices: list[int], similarity_matrix: np.ndarray, max_size: int) -> list[list[int]]:
    """
    v8 그룹은 이미 max_size 이하이지만,
    로컬 처리의 방어 로직으로 유지합니다.
    """
    if len(indices) <= max_size:
        return [sorted(indices)]
    remaining = set(indices)
    chunks = []
    while remaining:
        seed = max(remaining, key=lambda index: sum((float(similarity_matrix[index, other]) for other in remaining if other != index)))
        group = [seed]
        remaining.remove(seed)
        while remaining and len(group) < max_size:
            ranked = sorted(remaining, key=lambda index: min((float(similarity_matrix[index, member]) for member in group)), reverse=True)
            best = ranked[0]
            if min((float(similarity_matrix[best, member]) for member in group)) < GROUP_MIN_PAIR_SIMILARITY:
                break
            group.append(best)
            remaining.remove(best)
        chunks.append(sorted(group))
    return chunks


In [10]:
def scope_text(item: dict) -> str:
    return f"query: 요구사항명: {item.get('requirement_name', item.get('name', ''))}\n상세내용: {item.get('requirement_detail', item.get('description', ''))}"

def build_scope_embedding_cache(scope_reference_requirements: list[dict]) -> dict:
    valid = [dict(item) for item in scope_reference_requirements if isinstance(item, dict)]
    global_scope = [item for item in valid if item.get('global_scope') is True]
    local_scope = [item for item in valid if item.get('global_scope') is not True]
    if local_scope:
        local_vectors = get_embedding_model().encode([scope_text(item) for item in local_scope], batch_size=EMBEDDING_BATCH_SIZE, show_progress_bar=False, normalize_embeddings=True, convert_to_numpy=True)
    else:
        local_vectors = np.empty((0, 0), dtype=np.float32)
    return {'global_scope': global_scope, 'local_scope': local_scope, 'local_vectors': local_vectors}

def select_scope_for_group(group_candidates: list[dict], group_embeddings: np.ndarray, scope_cache: dict) -> list[dict]:
    global_scope = scope_cache.get('global_scope', [])
    local_scope = scope_cache.get('local_scope', [])
    scope_vectors = scope_cache.get('local_vectors')
    if not local_scope:
        return list(global_scope)
    group_vector = group_embeddings.mean(axis=0)
    norm = float(np.linalg.norm(group_vector))
    if norm > 0:
        group_vector = group_vector / norm
    scores = scope_vectors @ group_vector
    ranked = np.argsort(-scores)
    selected = []
    for scope_index in ranked[:SCOPE_TOP_K]:
        if float(scores[int(scope_index)]) >= SCOPE_SIM_THRESHOLD:
            selected.append(local_scope[int(scope_index)])
    return list(global_scope) + selected

def task3_payload_candidate(item: dict) -> dict:
    return {'task2_id': item['task2_id'], 'merge_decision': item.get('merge_decision', 'KEPT'), 'merged_from': dedupe_preserve(item.get('merged_from', [])), 'reference_context_ids': dedupe_preserve(item.get('reference_context_ids', [])), 'action_type': item.get('action_type', '미지정'), 'requirement_name': item.get('requirement_name', ''), 'requirement_detail': item.get('requirement_detail', ''), 'source_requirement_ids': dedupe_preserve(item.get('source_requirement_ids', []))}

def make_task3_user_obj(doc_id: str, candidates: list[dict], scope_reference_requirements: list[dict]) -> dict:
    return {'task_type': TASK3, 'document_id': doc_id, 'candidate_requirements': [task3_payload_candidate(item) for item in candidates], 'scope_reference_requirements': scope_reference_requirements}

def map_ids_to_original_lineage(ids, input_lineage_map: dict[str, list[str]]) -> list[str]:
    result = []
    for item_id in dedupe_preserve(ids):
        result.extend(input_lineage_map.get(item_id, [item_id]))
    return dedupe_preserve(result)

def resolve_stage3_lineage(finals: list[dict], relations: list[dict], input_candidates: list[dict]) -> tuple[list[dict], list[dict]]:
    input_lineage_map = {item['task2_id']: dedupe_preserve(item.get('lineage_task2_ids', [item['task2_id']])) for item in input_candidates}
    finals_by_gold = {}
    for final in finals:
        final['source_task2_ids'] = map_ids_to_original_lineage(final.get('source_task2_ids', []), input_lineage_map)
        finals_by_gold[str(final.get('gold_id', '')).strip()] = final
    normalized_relations = []
    for raw_relation in relations:
        if not isinstance(raw_relation, dict):
            continue
        relation = dict(raw_relation)
        relation['comparison_ids'] = map_ids_to_original_lineage(relation.get('comparison_ids', []), input_lineage_map)
        relation['excluded_or_absorbed_ids'] = map_ids_to_original_lineage(relation.get('excluded_or_absorbed_ids', []), input_lineage_map)
        target = finals_by_gold.get(str(relation.get('gold_id', '')).strip())
        if target is not None:
            target['source_task2_ids'] = dedupe_preserve(target.get('source_task2_ids', []) + relation['comparison_ids'] + relation['excluded_or_absorbed_ids'])
        normalized_relations.append(relation)
    return (finals, normalized_relations)

def expected_group_lineage_ids(input_candidates: list[dict]) -> set[str]:
    return {original_id for candidate in input_candidates for original_id in dedupe_preserve(candidate.get('lineage_task2_ids', [candidate['task2_id']]))}

def covered_group_lineage_ids(finals: list[dict]) -> set[str]:
    covered_ids = set()
    for final in finals:
        covered_ids.update(dedupe_preserve(final.get('source_task2_ids', [])))
    return covered_ids

def find_missing_group_coverage(input_candidates: list[dict], finals: list[dict], relations: list[dict] | None=None) -> list[str]:
    expected_ids = expected_group_lineage_ids(input_candidates)
    covered_ids = covered_group_lineage_ids(finals)
    return sorted(expected_ids - covered_ids)

def validate_final_task2_assignment(candidates: list[dict], finals: list[dict], relations: list[dict]) -> dict:
    expected_ids = {item['task2_id'] for item in candidates}
    final_gold_ids = {str(item.get('gold_id', '')).strip() for item in finals}
    if '' in final_gold_ids:
        raise ValueError('빈 gold_id가 있습니다.')
    orphan_relations = [relation for relation in relations if str(relation.get('gold_id', '')).strip() not in final_gold_ids]
    if orphan_relations:
        raise ValueError(f'존재하지 않는 GOLD를 참조하는 관계 판정이 있습니다: {orphan_relations[:3]}')
    assignment = defaultdict(list)
    unknown_final_ids = set()
    for final in finals:
        gold_id = str(final['gold_id']).strip()
        for task2_id in dedupe_preserve(final.get('source_task2_ids', [])):
            if task2_id not in expected_ids:
                unknown_final_ids.add(task2_id)
            assignment[task2_id].append(gold_id)
    unknown_relation_ids = set()
    for relation in relations:
        for task2_id in dedupe_preserve(relation.get('comparison_ids', [])) + dedupe_preserve(relation.get('excluded_or_absorbed_ids', [])):
            if task2_id not in expected_ids:
                unknown_relation_ids.add(task2_id)
    if unknown_final_ids:
        raise ValueError(f'최종 GOLD의 알 수 없는 TASK2 ID: {sorted(unknown_final_ids)}')
    if unknown_relation_ids:
        raise ValueError(f'관계 판정의 알 수 없는 TASK2 ID: {sorted(unknown_relation_ids)}')
    missing = sorted((task2_id for task2_id in expected_ids if not assignment[task2_id]))
    duplicated = {task2_id: gold_ids for task2_id, gold_ids in assignment.items() if len(set(gold_ids)) > 1}
    if missing:
        raise ValueError(f'최종 GOLD에 포함되지 않은 TASK2 ID: {missing}')
    if duplicated:
        raise ValueError(f'여러 GOLD에 중복 배정된 TASK2 ID: {duplicated}')
    return {'expected_task2_count': len(expected_ids), 'assigned_task2_count': len(assignment), 'missing_task2_count': 0, 'duplicate_assignment_count': 0, 'orphan_relation_count': 0}

def restore_missing_group_coverage(group_doc_id: str, input_candidates: list[dict], finals: list[dict], relations: list[dict], raw_log_path: Path) -> tuple[list[dict], list[dict], list[dict]]:
    """
    모델 출력에서 완전히 사라진 입력 후보만 독립 유지로 복구합니다.

    중복/병합 여부를 임의로 판단하지 않으며, 누락으로 인한 데이터
    손실을 막는 보수적 fallback입니다.
    """
    missing_ids = find_missing_group_coverage(input_candidates, finals, relations)
    if not missing_ids:
        return (finals, relations, [])
    if not TASK3_COVERAGE_FALLBACK_ENABLED:
        raise ValueError(f'TASK3 그룹 계보 누락: {missing_ids}')
    missing_set = set(missing_ids)
    existing_gold_ids = {str(item.get('gold_id', '')).strip() for item in finals if isinstance(item, dict)}
    fallback_records = []
    fallback_no = 1
    for candidate in input_candidates:
        candidate_lineage = dedupe_preserve(candidate.get('lineage_task2_ids', [candidate['task2_id']]))
        missing_for_candidate = [lineage_id for lineage_id in candidate_lineage if lineage_id in missing_set]
        if not missing_for_candidate:
            continue
        while True:
            fallback_gold_id = f'GOLD-COVERAGE-{fallback_no:03d}'
            fallback_no += 1
            if fallback_gold_id not in existing_gold_ids:
                break
        existing_gold_ids.add(fallback_gold_id)
        fallback_final = {'gold_id': fallback_gold_id, 'action_type': candidate.get('action_type', '미지정'), 'requirement_name': candidate.get('requirement_name', ''), 'requirement_detail': candidate.get('requirement_detail', ''), 'source_task2_ids': candidate_lineage, 'source_atomic_ids': dedupe_preserve(candidate.get('merged_from', [])), 'sources': dedupe_preserve(candidate.get('source_requirement_ids', [])), 'processing_type': 'KEPT', 'merge_basis': 'TASK3 유사 그룹 출력에서 해당 입력 계보가 최종 요구사항과 관계 판정 모두에서 누락되어 데이터 손실 방지를 위해 입력 요구사항을 독립 요구사항으로 보존함'}
        fallback_relation = {'relation_type': 'UNIQUE', 'processing_result': 'KEPT', 'comparison_ids': candidate_lineage, 'excluded_or_absorbed_ids': [], 'gold_id': fallback_gold_id, 'preserved_conditions': ['TASK3 그룹 출력 계보 누락으로 원문 전체 보존'], 'rationale': 'COVERAGE_FALLBACK_KEPT: 모델 출력에서 입력 후보가 누락되어 임의 병합 없이 독립 유지함'}
        finals.append(fallback_final)
        relations.append(fallback_relation)
        fallback_records.append({'group_document_id': group_doc_id, 'fallback_gold_id': fallback_gold_id, 'current_candidate_id': candidate.get('task2_id'), 'candidate_lineage': candidate_lineage, 'missing_lineage_ids': missing_for_candidate, 'action': 'COVERAGE_FALLBACK_KEPT', 'requirement_name': candidate.get('requirement_name', '')})
    still_missing = find_missing_group_coverage(input_candidates, finals, relations)
    if still_missing:
        raise ValueError(f'TASK3 계보 fallback 후에도 누락: {still_missing}')
    repair_path = raw_log_path.with_name(f'{raw_log_path.stem}_coverage_repair.json')
    dump_json(repair_path, {'group_document_id': group_doc_id, 'missing_before_repair': missing_ids, 'fallback_count': len(fallback_records), 'fallback_records': fallback_records})
    print(f'[TASK3 계보 안전 복구] group={group_doc_id}, missing={missing_ids}, fallback_kept={len(fallback_records)}', flush=True)
    return (finals, relations, fallback_records)

def validate_group_coverage(input_candidates: list[dict], finals: list[dict], relations: list[dict]) -> None:
    missing = find_missing_group_coverage(input_candidates, finals, relations)
    if missing:
        raise ValueError(f'TASK3 그룹 계보 누락: {missing}')

def run_stage3_group(group_doc_id: str, group_candidates: list[dict], selected_scope: list[dict], raw_log_path: Path) -> tuple[list[dict], list[dict], list[dict]]:
    if not group_candidates:
        raise ValueError('TASK3 유사 그룹 후보가 비어 있습니다.')
    if len(group_candidates) > TASK3_MAX_GROUP_SIZE:
        raise ValueError(f'TASK3 유사 그룹 크기 초과: {len(group_candidates)} > {TASK3_MAX_GROUP_SIZE}')
    user_obj = make_task3_user_obj(group_doc_id, group_candidates, selected_scope)
    obj, _ = run_task(TASK3, user_obj, raw_log_path=raw_log_path)
    finals = [dict(item) for item in obj['final_requirements']]
    relations = [dict(item) for item in obj.get('relation_decisions', []) if isinstance(item, dict)]
    local_gold_ids = [str(item.get('gold_id', '')).strip() for item in finals]
    if not all(local_gold_ids) or len(local_gold_ids) != len(set(local_gold_ids)):
        raise ValueError('TASK3 그룹 출력의 gold_id가 비어 있거나 중복되었습니다.')
    finals, relations = resolve_stage3_lineage(finals, relations, group_candidates)
    finals, relations, coverage_fallback_records = restore_missing_group_coverage(group_doc_id, group_candidates, finals, relations, raw_log_path)
    validate_group_coverage(group_candidates, finals, relations)
    validate_final_task2_assignment([{**candidate, 'task2_id': lineage_id} for candidate in group_candidates for lineage_id in dedupe_preserve(candidate.get('lineage_task2_ids', [candidate['task2_id']]))], finals, relations)
    return (finals, relations, coverage_fallback_records)

def final_to_local_candidate(final: dict, local_id: str) -> dict:
    return {'task2_id': local_id, 'merge_decision': 'KEPT', 'merged_from': dedupe_preserve(final.get('source_atomic_ids', [])), 'reference_context_ids': [], 'action_type': final.get('action_type', '미지정'), 'requirement_name': final.get('requirement_name', ''), 'requirement_detail': final.get('requirement_detail', ''), 'source_requirement_ids': dedupe_preserve(final.get('sources', [])), 'lineage_task2_ids': dedupe_preserve(final.get('source_task2_ids', []))}

def embed_candidate_list(candidates: list[dict]) -> np.ndarray:
    return get_embedding_model().encode([embedding_text(item) for item in candidates], batch_size=EMBEDDING_BATCH_SIZE, show_progress_bar=False, normalize_embeddings=True, convert_to_numpy=True)

def process_similarity_component(doc_id: str, component_no: int, component_candidates: list[dict], scope_cache: dict, raw_component_dir: Path) -> tuple[list[dict], list[dict], dict]:
    current_candidates = [{**dict(item), 'lineage_task2_ids': dedupe_preserve(item.get('lineage_task2_ids', [item['task2_id']]))} for item in component_candidates]
    trace_rounds = []
    for round_no in range(1, TASK3_MAX_LOCAL_ROUNDS + 1):
        current_embeddings = embed_candidate_list(current_candidates)
        current_similarity = current_embeddings @ current_embeddings.T
        current_indices = list(range(len(current_candidates)))
        chunks = split_indices_by_similarity(current_indices, current_similarity, TASK3_MAX_GROUP_SIZE)
        round_finals = []
        round_relations = []
        round_fallback_records = []
        print(f'[TASK3 컴포넌트] component={component_no}, round={round_no}, candidates={len(current_candidates)}, chunks={len(chunks)}', flush=True)
        for chunk_no, chunk_indices in enumerate(chunks, start=1):
            chunk_candidates = [current_candidates[index] for index in chunk_indices]
            chunk_embeddings = current_embeddings[chunk_indices]
            selected_scope = select_scope_for_group(chunk_candidates, chunk_embeddings, scope_cache)
            group_doc_id = f'{doc_id}-EMB-C{component_no:03d}-R{round_no:02d}-G{chunk_no:03d}'
            finals, relations, fallback_records = run_stage3_group(group_doc_id, chunk_candidates, selected_scope, raw_component_dir / f'round_{round_no:02d}' / f'group_{chunk_no:03d}.json')
            round_finals.extend(finals)
            round_relations.extend(relations)
            round_fallback_records.extend(fallback_records)
        trace_rounds.append({'round': round_no, 'input_candidate_count': len(current_candidates), 'chunk_count': len(chunks), 'output_final_count': len(round_finals), 'fallback_count': len(round_fallback_records)})
        if len(chunks) == 1:
            return (round_finals, round_relations, {'component_no': component_no, 'initial_count': len(component_candidates), 'rounds': trace_rounds, 'status': 'COMPLETED', 'fallback_records': round_fallback_records})
        next_candidates = [final_to_local_candidate(final, f'H3-C{component_no:03d}-R{round_no:02d}-{index:04d}') for index, final in enumerate(round_finals, start=1)]
        if not next_candidates:
            raise RuntimeError('TASK3 로컬 다음 라운드 후보가 0건입니다.')
        current_candidates = next_candidates
    print(f'[경고] TASK3 컴포넌트 {component_no}: 최대 로컬 라운드 {TASK3_MAX_LOCAL_ROUNDS}회에 도달했습니다. 마지막 소규모 그룹 결과를 결합합니다.', flush=True)
    return (round_finals, round_relations, {'component_no': component_no, 'initial_count': len(component_candidates), 'rounds': trace_rounds, 'status': 'MAX_LOCAL_ROUNDS_REACHED', 'fallback_records': round_fallback_records})

def singleton_to_final(candidate: dict, temp_gold_id: str) -> tuple[dict, dict]:
    final = {'gold_id': temp_gold_id, 'action_type': candidate.get('action_type', '미지정'), 'requirement_name': candidate.get('requirement_name', ''), 'requirement_detail': candidate.get('requirement_detail', ''), 'source_task2_ids': [candidate['task2_id']], 'source_atomic_ids': dedupe_preserve(candidate.get('merged_from', [])), 'sources': dedupe_preserve(candidate.get('source_requirement_ids', [])), 'processing_type': 'KEPT', 'merge_basis': '문서 전체 TASK2 후보 임베딩 비교에서 설정 임계값 이상의 유사 후보가 없어 TASK2 내용을 그대로 보존함'}
    relation = {'relation_type': 'UNIQUE', 'processing_result': 'KEPT', 'comparison_ids': [candidate['task2_id']], 'excluded_or_absorbed_ids': [], 'gold_id': temp_gold_id, 'preserved_conditions': [], 'rationale': '임베딩 유사 후보 없음'}
    return (final, relation)


In [11]:
def validate_user_input(input_obj: Any, input_path: Path) -> dict:
    if not isinstance(input_obj, dict):
        raise TypeError(f'{input_path.name}: 최상위 JSON은 객체여야 합니다.')
    required_top = {'document_id', 'document_name', 'functional_requirements'}
    missing_top = required_top - set(input_obj)
    if missing_top:
        raise ValueError(f'{input_path.name}: 최상위 필드 누락 {sorted(missing_top)}')
    requirements = input_obj['functional_requirements']
    if not isinstance(requirements, list) or not requirements:
        raise ValueError('functional_requirements는 1건 이상의 배열이어야 합니다.')
    required_fur = {'requirement_id', 'requirement_name', 'requirement_type', 'requirement_definition', 'requirement_detail'}
    seen_ids = set()
    for index, fur in enumerate(requirements):
        if not isinstance(fur, dict):
            raise TypeError(f'functional_requirements[{index}]는 객체여야 합니다.')
        missing = required_fur - set(fur)
        if missing:
            raise ValueError(f'functional_requirements[{index}] 필드 누락={sorted(missing)}')
        fur_id = str(fur['requirement_id']).strip()
        if not fur_id:
            raise ValueError(f'functional_requirements[{index}].requirement_id가 비어 있습니다.')
        if fur_id in seen_ids:
            raise ValueError(f'중복 requirement_id: {fur_id}')
        seen_ids.add(fur_id)
        if not str(fur['requirement_detail']).strip():
            raise ValueError(f'{fur_id}.requirement_detail이 비어 있습니다.')
    scope_refs = input_obj.get('scope_reference_requirements', [])
    if not isinstance(scope_refs, list):
        raise TypeError('scope_reference_requirements는 배열이어야 합니다.')
    return input_obj

def discover_user_test_files() -> list[Path]:
    valid_files = []
    for path in sorted(USER_TEST_INPUT_DIR.glob(USER_INPUT_GLOB)):
        try:
            obj = load_json(path)
        except Exception:
            continue
        if isinstance(obj, dict) and isinstance(obj.get('functional_requirements'), list):
            valid_files.append(path)
    return valid_files

def enrich_final_lineage(final: dict, task2_by_id: dict[str, dict]) -> None:
    source_task2_ids = dedupe_preserve(final.get('source_task2_ids', []))
    source_atomic_ids = dedupe_preserve(final.get('source_atomic_ids', []))
    sources = dedupe_preserve(final.get('sources', []))
    for task2_id in source_task2_ids:
        source_candidate = task2_by_id.get(task2_id)
        if source_candidate is None:
            continue
        source_atomic_ids.extend(source_candidate.get('merged_from', []))
        sources.extend(source_candidate.get('source_requirement_ids', []))
    final['source_task2_ids'] = dedupe_preserve(source_task2_ids)
    final['source_atomic_ids'] = dedupe_preserve(source_atomic_ids)
    final['sources'] = dedupe_preserve(sources)

def clear_task3_outputs_only(doc_out_dir: Path) -> None:
    targets = [doc_out_dir / 'task3_embedding', doc_out_dir / 'raw' / 'task3_embedding', doc_out_dir / 'task3_final.json', doc_out_dir / '예측_GOLD.json', doc_out_dir / '예측_GOLD.csv', doc_out_dir / 'quality_report.json']
    for target in targets:
        if target.is_dir():
            shutil.rmtree(target)
        elif target.exists():
            target.unlink()

def build_run_manifest(input_obj: dict) -> dict:
    prompt_hashes = {task_type: text_sha256(SYSTEM_PROMPTS[task_type]) for task_type in (TASK1, TASK2, TASK3)}
    task1_signature = {'pipeline_version': PIPELINE_VERSION, 'input_sha256': canonical_json_sha256(input_obj), 'base_model': BASE_MODEL, 'stage1_adapter': STAGE1_ADAPTER_REPO, 'task1_prompt_sha256': prompt_hashes[TASK1]}
    task2_signature = {**task1_signature, 'task2_prompt_sha256': prompt_hashes[TASK2]}
    task3_signature = {**task2_signature, 'stage3_adapter': STAGE3_ADAPTER_REPO, 'task3_prompt_sha256': prompt_hashes[TASK3], 'embedding_model': EMBEDDING_MODEL_NAME, 'embedding_settings': {'high': HIGH_SIM_THRESHOLD, 'review': REVIEW_SIM_THRESHOLD, 'adaptive_floor_max': ADAPTIVE_FLOOR_MAX, 'pair_quantile': PAIR_SIMILARITY_QUANTILE, 'mutual_top_k': MUTUAL_TOP_K, 'name_jaccard': NAME_JACCARD_THRESHOLD, 'lexical_cosine': LEXICAL_COSINE_THRESHOLD, 'lexical_name_jaccard': LEXICAL_NAME_JACCARD_THRESHOLD, 'group_min_pair': GROUP_MIN_PAIR_SIMILARITY, 'max_group_size': TASK3_MAX_GROUP_SIZE}}
    return {'pipeline_version': PIPELINE_VERSION, 'created_at_utc': datetime.now(timezone.utc).isoformat(), 'document_id': str(input_obj['document_id']).strip(), 'input_sha256': task1_signature['input_sha256'], 'prompt_hashes': prompt_hashes, 'task1_signature': canonical_json_sha256(task1_signature), 'task2_signature': canonical_json_sha256(task2_signature), 'task3_signature': canonical_json_sha256(task3_signature), 'contracts': {'base_model': BASE_MODEL, 'stage1_adapter': STAGE1_ADAPTER_REPO, 'stage3_adapter': STAGE3_ADAPTER_REPO, 'embedding_model': EMBEDDING_MODEL_NAME}, 'progress': {'task1_completed_furs': [], 'task2_completed_furs': [], 'task3_status': 'NOT_STARTED'}}

def manifest_signature_matches(existing: dict | None, current: dict, stage: str) -> bool:
    if not isinstance(existing, dict):
        return False
    key = f'{stage}_signature'
    return existing.get(key) == current.get(key)

def update_manifest_progress(manifest_path: Path, manifest: dict, *, task1_completed_furs: list[str] | None=None, task2_completed_furs: list[str] | None=None, task3_status: str | None=None) -> None:
    progress = manifest.setdefault('progress', {})
    if task1_completed_furs is not None:
        progress['task1_completed_furs'] = dedupe_preserve(task1_completed_furs)
    if task2_completed_furs is not None:
        progress['task2_completed_furs'] = dedupe_preserve(task2_completed_furs)
    if task3_status is not None:
        progress['task3_status'] = task3_status
    progress['updated_at_utc'] = datetime.now(timezone.utc).isoformat()
    dump_json(manifest_path, manifest)

def validate_saved_task1_subset(all_atomics_by_fur: dict, expected_fur_ids: set[str]) -> None:
    unknown = set(all_atomics_by_fur) - expected_fur_ids
    if unknown:
        raise ValueError(f'저장된 TASK1 결과에 현재 입력에 없는 FUR가 있습니다: {sorted(unknown)}')
    for fur_id, atomics in all_atomics_by_fur.items():
        if not isinstance(atomics, list):
            raise TypeError(f'{fur_id}: 저장 TASK1 결과가 배열이 아닙니다.')
        validate_task1_result(fur_id, atomics)

def load_task2_checkpoint(checkpoint_path: Path, fur_id: str, atomics: list[dict]) -> list[dict]:
    obj = load_json(checkpoint_path)
    if not isinstance(obj, dict):
        raise TypeError(f'{checkpoint_path}: TASK2 체크포인트가 객체가 아닙니다.')
    if str(obj.get('source_requirement_id', '')).strip() != fur_id:
        raise ValueError(f'{checkpoint_path}: FUR ID가 현재 입력과 다릅니다.')
    normalized = obj.get('normalized_requirements')
    if not isinstance(normalized, list) or not normalized:
        raise ValueError(f'{checkpoint_path}: normalized_requirements가 비어 있습니다.')
    normalized = normalize_task2_local_output(fur_id, normalized)
    validate_task2_atomic_coverage(atomics, normalized)
    return normalized

def run_pipeline_from_user_file(input_path: Path) -> tuple[list[dict], list[dict], dict]:
    input_path = Path(input_path)
    input_obj = validate_user_input(load_json(input_path), input_path)
    doc_id = str(input_obj['document_id']).strip()
    document_name = str(input_obj['document_name']).strip()
    functional_requirements = input_obj['functional_requirements']
    scope_reference_requirements = input_obj.get('scope_reference_requirements', [])
    expected_fur_ids = {str(fur['requirement_id']).strip() for fur in functional_requirements}
    doc_out_dir = OUT_DIR / doc_id
    if doc_out_dir.exists():
        print(f'[신규 실행] {doc_id}: 기존 문서 산출물을 삭제하고 TASK1부터 시작합니다.', flush=True)
        shutil.rmtree(doc_out_dir)
    manifest_path = doc_out_dir / 'run_manifest.json'
    current_manifest = build_run_manifest(input_obj)
    existing_manifest = None
    task1_compatible = task2_compatible = task3_compatible = False
    doc_out_dir.mkdir(parents=True, exist_ok=True)
    raw_dir = doc_out_dir / 'raw'
    embedding_dir = doc_out_dir / 'task3_embedding'
    task1_path = doc_out_dir / 'task1_atomics_by_fur.json'
    task2_path = doc_out_dir / 'task2_candidates.json'
    task2_checkpoint_dir = doc_out_dir / 'task2_by_fur'
    raw_dir.mkdir(parents=True, exist_ok=True)
    task2_checkpoint_dir.mkdir(parents=True, exist_ok=True)
    if task1_compatible and task2_compatible and (not task3_compatible):
        clear_task3_outputs_only(doc_out_dir)
    manifest = current_manifest
    if isinstance(existing_manifest, dict):
        manifest['progress'] = existing_manifest.get('progress', manifest['progress'])
    dump_json(manifest_path, manifest)
    dump_json(doc_out_dir / 'input_user_document.json', input_obj)
    trace = {'document_id': doc_id, 'document_name': document_name, 'input_file': str(input_path), 'input_fur_count': len(functional_requirements), 'pipeline': 'TASK1_TASK2_TASK3_EMBEDDING_GROUP_V10_4_RELATION_REPAIR_EVALUATION', 'pipeline_version': PIPELINE_VERSION, 'manifest_signatures': {'task1': manifest['task1_signature'], 'task2': manifest['task2_signature'], 'task3': manifest['task3_signature']}, 'gpu': torch.cuda.get_device_name(0), 'embedding_device': EMBEDDING_DEVICE}
    if task1_compatible and task1_path.exists():
        all_atomics_by_fur = load_json(task1_path)
        if not isinstance(all_atomics_by_fur, dict):
            raise TypeError('저장된 TASK1 결과가 객체가 아닙니다.')
        validate_saved_task1_subset(all_atomics_by_fur, expected_fur_ids)
    else:
        all_atomics_by_fur = {}
    reused_task1_furs = []
    generated_task1_furs = []
    for fur in tqdm(functional_requirements, desc=f'{doc_id} TASK1', leave=False):
        fur_id = str(fur['requirement_id']).strip()
        existing_atomics = all_atomics_by_fur.get(fur_id)
        if existing_atomics:
            validate_task1_result(fur_id, existing_atomics)
            reused_task1_furs.append(fur_id)
            continue
        atomics = stage_task1(doc_id, document_name, fur, raw_log_path=raw_dir / 'task1' / f'{safe_file_component(fur_id)}.json')
        all_atomics_by_fur[fur_id] = atomics
        generated_task1_furs.append(fur_id)
        dump_json(task1_path, all_atomics_by_fur)
        update_manifest_progress(manifest_path, manifest, task1_completed_furs=list(all_atomics_by_fur))
    if set(all_atomics_by_fur) != expected_fur_ids:
        raise RuntimeError(f'TASK1 완료 FUR 집합이 입력과 다릅니다. missing={sorted(expected_fur_ids - set(all_atomics_by_fur))}, unknown={sorted(set(all_atomics_by_fur) - expected_fur_ids)}')
    dump_json(task1_path, all_atomics_by_fur)
    trace['task1_atomic_total'] = sum((len(items) for items in all_atomics_by_fur.values()))
    trace['task1_reused_fur_count'] = len(reused_task1_furs)
    trace['task1_generated_fur_count'] = len(generated_task1_furs)
    local_candidates = []
    reused_task2_furs = []
    generated_task2_furs = []
    for fur in tqdm(functional_requirements, desc=f'{doc_id} TASK2', leave=False):
        fur_id = str(fur['requirement_id']).strip()
        atomics = all_atomics_by_fur[fur_id]
        checkpoint_path = task2_checkpoint_dir / f'{safe_file_component(fur_id)}.json'
        normalized = None
        if task2_compatible and checkpoint_path.exists():
            try:
                normalized = load_task2_checkpoint(checkpoint_path, fur_id, atomics)
                reused_task2_furs.append(fur_id)
            except Exception as exc:
                print(f'[TASK2 체크포인트 재생성] {fur_id}: {exc}', flush=True)
                checkpoint_path.unlink(missing_ok=True)
        if normalized is None:
            normalized = stage_task2(doc_id, fur_id, atomics, raw_log_path=raw_dir / 'task2' / f'{safe_file_component(fur_id)}.json')
            generated_task2_furs.append(fur_id)
            dump_json(checkpoint_path, {'task_type': TASK2, 'document_id': doc_id, 'source_requirement_id': fur_id, 'atomic_input_sha256': canonical_json_sha256(atomics), 'normalized_requirement_count': len(normalized), 'normalized_requirements': normalized})
        local_candidates.extend(normalized)
        update_manifest_progress(manifest_path, manifest, task2_completed_furs=reused_task2_furs + generated_task2_furs)
    if not local_candidates:
        raise RuntimeError(f'{doc_id}: TASK2 후보가 0건입니다.')
    candidates = normalize_task2_global_ids(local_candidates)
    validate_document_task2_lineage(all_atomics_by_fur, candidates)
    dump_json(task2_path, {'task_type': TASK2, 'document_id': doc_id, 'candidate_requirement_count': len(candidates), 'candidate_requirements': candidates})
    trace['task2_candidate_total'] = len(candidates)
    trace['task2_reused_fur_count'] = len(reused_task2_furs)
    trace['task2_generated_fur_count'] = len(generated_task2_furs)
    clear_task3_outputs_only(doc_out_dir)
    update_manifest_progress(manifest_path, manifest, task3_status='RUNNING')
    similarity_result = build_similarity_groups(doc_id, candidates, embedding_dir)
    embeddings = similarity_result['embeddings']
    similar_components = similarity_result['similar_components']
    singleton_indices = similarity_result['singleton_indices']
    scope_cache = build_scope_embedding_cache(scope_reference_requirements)
    task2_by_id = {item['task2_id']: item for item in candidates}
    task2_order = {item['task2_id']: index for index, item in enumerate(candidates)}
    combined_finals = []
    combined_relations = []
    component_traces = []
    all_fallback_records = []
    global_scope = scope_cache.get('global_scope', [])
    for singleton_no, candidate_index in enumerate(singleton_indices, start=1):
        candidate = candidates[candidate_index]
        if APPLY_GLOBAL_SCOPE_TO_SINGLETONS and global_scope:
            group_doc_id = f'{doc_id}-SCOPE-S{singleton_no:04d}'
            finals, relations, fallback_records = run_stage3_group(group_doc_id, [{**candidate, 'lineage_task2_ids': [candidate['task2_id']]}], global_scope, raw_dir / 'task3_embedding' / 'singletons' / f'singleton_{singleton_no:04d}.json')
            all_fallback_records.extend(fallback_records)
            local_gold_map = {}
            for local_index, final in enumerate(finals, start=1):
                final = dict(final)
                old_gold_id = str(final.get('gold_id', '')).strip()
                temp_gold_id = f'TMP-S{singleton_no:04d}-{local_index:02d}'
                if old_gold_id:
                    local_gold_map[old_gold_id] = temp_gold_id
                final['gold_id'] = temp_gold_id
                enrich_final_lineage(final, task2_by_id)
                combined_finals.append(final)
            for raw_relation in relations:
                relation = dict(raw_relation)
                old_gold_id = str(relation.get('gold_id', '')).strip()
                if old_gold_id in local_gold_map:
                    relation['gold_id'] = local_gold_map[old_gold_id]
                combined_relations.append(relation)
        else:
            temp_gold_id = f'TMP-S-{singleton_no:04d}'
            final, relation = singleton_to_final(candidate, temp_gold_id)
            combined_finals.append(final)
            combined_relations.append(relation)
    for component_no, component_indices in enumerate(similar_components, start=1):
        component_candidates = [candidates[index] for index in component_indices]
        finals, relations, component_trace = process_similarity_component(doc_id, component_no, component_candidates, scope_cache, raw_dir / 'task3_embedding' / f'component_{component_no:03d}')
        all_fallback_records.extend(component_trace.get('fallback_records', []))
        local_gold_map = {}
        for local_index, final in enumerate(finals, start=1):
            final = dict(final)
            old_gold_id = str(final.get('gold_id', '')).strip()
            temp_gold_id = f'TMP-C{component_no:03d}-{local_index:04d}'
            if old_gold_id:
                local_gold_map[old_gold_id] = temp_gold_id
            final['gold_id'] = temp_gold_id
            enrich_final_lineage(final, task2_by_id)
            combined_finals.append(final)
        for raw_relation in relations:
            relation = dict(raw_relation)
            old_gold_id = str(relation.get('gold_id', '')).strip()
            if old_gold_id in local_gold_map:
                relation['gold_id'] = local_gold_map[old_gold_id]
            combined_relations.append(relation)
        component_traces.append(component_trace)
    final_by_temp_id = {final['gold_id']: final for final in combined_finals}
    for relation in combined_relations:
        target = final_by_temp_id.get(relation.get('gold_id'))
        if target is None:
            raise ValueError(f"존재하지 않는 임시 GOLD를 참조하는 관계 판정: {relation.get('gold_id')}")
        relation_ids = dedupe_preserve(relation.get('comparison_ids', [])) + dedupe_preserve(relation.get('excluded_or_absorbed_ids', []))
        target['source_task2_ids'] = dedupe_preserve(target.get('source_task2_ids', []) + relation_ids)
        enrich_final_lineage(target, task2_by_id)

    def final_sort_key(final: dict):
        positions = [task2_order[task2_id] for task2_id in final.get('source_task2_ids', []) if task2_id in task2_order]
        return (min(positions) if positions else 10 ** 9, final.get('requirement_name', ''))
    combined_finals.sort(key=final_sort_key)
    temp_to_gold = {}
    for index, final in enumerate(combined_finals, start=1):
        old_gold_id = final['gold_id']
        new_gold_id = f'GOLD-{index:03d}'
        temp_to_gold[old_gold_id] = new_gold_id
        final['gold_id'] = new_gold_id
    for relation in combined_relations:
        old_gold_id = relation.get('gold_id')
        if old_gold_id not in temp_to_gold:
            raise ValueError(f'관계 판정 임시 GOLD ID 매핑 실패: {old_gold_id}')
        relation['gold_id'] = temp_to_gold[old_gold_id]
    assignment_quality = validate_final_task2_assignment(candidates, combined_finals, combined_relations)
    gold_ids = [final['gold_id'] for final in combined_finals]
    if len(gold_ids) != len(set(gold_ids)):
        raise RuntimeError('최종 GOLD ID 중복이 발생했습니다.')
    fallback_count = len(all_fallback_records)
    quality_status = 'PASS' if fallback_count == 0 else 'REVIEW_REQUIRED'
    trace.update({'embedding_model': EMBEDDING_MODEL_NAME, 'scope_global_count': len(global_scope), 'scope_local_count': len(scope_cache.get('local_scope', [])), 'similar_component_count': len(similar_components), 'similar_component_item_count': sum((len(component) for component in similar_components)), 'singleton_count': len(singleton_indices), 'task3_component_traces': component_traces, 'task3_fallback_count': fallback_count, 'quality_status': quality_status, 'assignment_quality': assignment_quality, 'task3_final_total': len(combined_finals)})
    final_output = {'task_type': TASK3, 'document_id': doc_id, 'final_requirement_count': len(combined_finals), 'final_requirements': combined_finals, 'relation_decisions': combined_relations, 'quality': {'status': quality_status, 'fallback_count': fallback_count, 'fallback_records': all_fallback_records, **assignment_quality}, 'trace': trace}
    dump_json(doc_out_dir / 'task3_final.json', final_output)
    dump_json(doc_out_dir / '예측_GOLD.json', final_output)
    dump_json(OUT_DIR / f'{doc_id}_예측_GOLD.json', final_output)
    dump_json(doc_out_dir / 'quality_report.json', final_output['quality'])
    summary_rows = []
    for final in combined_finals:
        summary_rows.append({'gold_id': final['gold_id'], 'action_type': final.get('action_type', ''), 'requirement_name': final.get('requirement_name', ''), 'requirement_detail': final.get('requirement_detail', ''), 'source_task2_ids': '; '.join(final.get('source_task2_ids', [])), 'source_atomic_ids': '; '.join(final.get('source_atomic_ids', [])), 'sources': '; '.join(final.get('sources', [])), 'processing_type': final.get('processing_type', ''), 'merge_basis': final.get('merge_basis', ''), 'quality_status': quality_status})
    pd.DataFrame(summary_rows).to_csv(doc_out_dir / '예측_GOLD.csv', index=False, encoding='utf-8-sig')
    update_manifest_progress(manifest_path, manifest, task1_completed_furs=sorted(expected_fur_ids), task2_completed_furs=sorted(expected_fur_ids), task3_status=quality_status)
    print('=' * 72)
    print('[문서 처리 완료]')
    print('document_id          :', doc_id)
    print('TASK1 atomic total   :', trace['task1_atomic_total'])
    print('TASK2 candidate total:', trace['task2_candidate_total'])
    print('TASK3 final GOLD     :', trace['task3_final_total'])
    print('Fallback count       :', fallback_count)
    print('Quality status       :', quality_status)
    print('결과 파일            :', doc_out_dir / '예측_GOLD.json')
    print('=' * 72)
    return (combined_finals, combined_relations, trace)


In [12]:
EVAL_TASK2_RETENTION_THRESHOLD = 0.72
EVAL_FUR_COVERAGE_THRESHOLD = 0.68
EVAL_NAME_DETAIL_THRESHOLD = 0.55

# 잔존 중복 평가는 TASK3 후보 검색보다 엄격하게 적용합니다.
# 임베딩 유사도만 높다는 이유로 같은 도메인의 별도 기능을 중복으로 판정하지 않습니다.
EVAL_RESIDUAL_STRONG_SIM = 0.965
EVAL_RESIDUAL_REVIEW_SIM = 0.940
EVAL_RESIDUAL_STRONG_NAME_JACCARD = 0.45
EVAL_RESIDUAL_REVIEW_NAME_JACCARD = 0.30

EVAL_OVERMERGE_PAIR_THRESHOLD = 0.65
EVAL_REFERENCE_MATCH_THRESHOLD = 0.80
EVAL_PASS_SCORE = 85.0
EVAL_REVIEW_SCORE = 70.0


def find_gold_final_path(doc_id: str, input_dir: Path | None=None) -> Path | None:
    roots = []
    if input_dir is not None: roots.append(Path(input_dir))
    roots.append(USER_TEST_INPUT_DIR)
    seen = set()
    for root in roots:
        if str(root) in seen: continue
        seen.add(str(root))
        for path in (root / f'{doc_id}_gold_final.json', root / f'{doc_id}_gold.json'):
            if path.exists(): return path
    return None


def extract_gold_items(obj: Any) -> list[dict]:
    if isinstance(obj, list):
        return [item for item in obj if isinstance(item, dict)]
    if isinstance(obj, dict):
        for key in ('final_requirements', 'gold_requirements', 'requirements'):
            value = obj.get(key)
            if isinstance(value, list):
                return [item for item in value if isinstance(item, dict)]
    raise ValueError('GOLD 요구사항 배열을 찾지 못했습니다.')


def _ratio(numerator: float, denominator: float, default: float=0.0) -> float:
    return float(numerator / denominator) if denominator else float(default)


def _mean(values: list[float], default: float=0.0) -> float:
    return float(np.mean(values)) if values else float(default)


def _round(value: Any, digits: int=6) -> Any:
    if value is None:
        return None
    return round(float(value), digits)


def _normalized_name(item: dict) -> str:
    return re.sub(r'\s+', '', str(item.get('requirement_name') or item.get('name') or '')).lower()


def _requirement_text(item: dict) -> str:
    name = str(item.get('requirement_name') or item.get('name') or '').strip()
    detail = str(item.get('requirement_detail') or item.get('description') or item.get('requirement_definition') or '').strip()
    action = str(item.get('action_type') or '').strip()
    return f'수행행위: {action}\n요구사항명: {name}\n상세내용: {detail}'


_DUPLICATE_BOILERPLATE_PATTERNS = (
    r'기능을\s*제공하여야\s*한다',
    r'제공하여야\s*한다',
    r'지원하여야\s*한다',
    r'포함하여야\s*한다',
    r'적용하여야\s*한다',
    r'할\s*수\s*있어야\s*한다',
    r'시스템은',
    r'사용자는',
)


def _normalize_action_type(item: dict) -> str:
    return re.sub(r'\s+', '', str(item.get('action_type', '')).strip().lower())


def _duplicate_evaluation_text(item: dict) -> str:
    """잔존 중복 평가용 텍스트. 반복 종결 표현을 제거해 도메인 유사도 과대평가를 줄입니다."""
    action = str(item.get('action_type', '')).strip()
    name = str(item.get('requirement_name') or item.get('name') or '').strip()
    detail = str(
        item.get('requirement_detail')
        or item.get('description')
        or item.get('requirement_definition')
        or ''
    ).strip()
    for pattern in _DUPLICATE_BOILERPLATE_PATTERNS:
        detail = re.sub(pattern, ' ', detail, flags=re.IGNORECASE)
    detail = re.sub(r'\s+', ' ', detail).strip()
    return f'수행행위: {action}\n요구사항명: {name}\n핵심내용: {detail}'


def _fur_text(item: dict) -> str:
    return f"요구사항명: {item.get('requirement_name', '')}\n정의: {item.get('requirement_definition', '')}\n상세내용: {item.get('requirement_detail', '')}"


def _encode_texts(texts: list[str]) -> np.ndarray:
    if not texts:
        return np.zeros((0, 1), dtype=np.float32)
    prefixed = [f'query: {text}' for text in texts]
    return get_embedding_model().encode(prefixed, batch_size=EMBEDDING_BATCH_SIZE, show_progress_bar=False, normalize_embeddings=True, convert_to_numpy=True)


def _greedy_matches(similarity_matrix: np.ndarray, threshold: float) -> list[dict]:
    if similarity_matrix.size == 0:
        return []
    pairs = [(float(similarity_matrix[left, right]), left, right) for left in range(similarity_matrix.shape[0]) for right in range(similarity_matrix.shape[1])]
    used_left, used_right, matches = set(), set(), []
    for score, left, right in sorted(pairs, reverse=True):
        if score < threshold:
            break
        if left in used_left or right in used_right:
            continue
        used_left.add(left); used_right.add(right)
        matches.append({'predicted_index': left, 'gold_index': right, 'semantic_similarity': _round(score)})
    return matches


def _load_task2_candidates(doc_out_dir: Path) -> list[dict]:
    obj = load_json(doc_out_dir / 'task2_candidates.json')
    if isinstance(obj, list):
        return [item for item in obj if isinstance(item, dict)]
    for key in ('candidate_requirements', 'normalized_requirements', 'task2_candidates', 'requirements'):
        value = obj.get(key) if isinstance(obj, dict) else None
        if isinstance(value, list):
            return [item for item in value if isinstance(item, dict)]
    raise ValueError('평가용 TASK2 후보 배열을 찾지 못했습니다.')


def _load_task1_atomics(doc_out_dir: Path) -> list[dict]:
    obj = load_json(doc_out_dir / 'task1_atomics_by_fur.json')
    if not isinstance(obj, dict):
        raise TypeError('평가용 TASK1 결과는 FUR별 객체여야 합니다.')
    return [item for values in obj.values() if isinstance(values, list) for item in values if isinstance(item, dict)]


def _save_csv(path: Path, rows: list[dict], columns: list[str] | None=None) -> None:
    frame = pd.DataFrame(rows, columns=columns) if columns else pd.DataFrame(rows)
    frame.to_csv(path, index=False, encoding='utf-8-sig')


def evaluate_document_output(input_path: Path, predicted: list[dict], relations: list[dict], trace: dict) -> dict:
    input_path = Path(input_path)
    input_obj = validate_user_input(load_json(input_path), input_path)
    doc_id = str(input_obj['document_id']).strip()
    doc_out_dir, eval_dir = OUT_DIR / doc_id, OUT_DIR / doc_id / 'evaluation'
    eval_dir.mkdir(parents=True, exist_ok=True)
    atomics = _load_task1_atomics(doc_out_dir)
    candidates = _load_task2_candidates(doc_out_dir)
    furs = input_obj['functional_requirements']
    fur_by_id = {str(item['requirement_id']).strip(): item for item in furs}
    atomic_by_id = {str(item.get('atomic_id', '')).strip(): item for item in atomics}
    task2_by_id = {str(item.get('task2_id', '')).strip(): item for item in candidates}
    final_by_id = {str(item.get('gold_id', '')).strip(): item for item in predicted}
    review_rows: list[dict] = []

    # 1) 입력→출력 수량 변화
    input_chars = sum(len(str(item.get('requirement_detail', ''))) for item in furs)
    output_chars = sum(len(str(item.get('requirement_detail', ''))) for item in predicted)
    counts = {'input_fur_count': len(furs), 'input_detail_character_count': input_chars, 'task1_atomic_count': len(atomics), 'task2_candidate_count': len(candidates), 'final_gold_count': len(predicted), 'relation_decision_count': len(relations), 'task1_decomposition_ratio': _round(_ratio(len(atomics), len(furs))), 'task2_reduction_ratio_from_task1': _round(1.0 - _ratio(len(candidates), len(atomics), 1.0)), 'task3_reduction_ratio_from_task2': _round(1.0 - _ratio(len(predicted), len(candidates), 1.0)), 'output_to_input_character_ratio': _round(_ratio(output_chars, input_chars))}
    _save_csv(eval_dir / '01_입력대비_출력변화.csv', [counts])

    # 2) 구조·필드·문장 품질
    required_fields = ('gold_id', 'action_type', 'requirement_name', 'requirement_detail', 'source_task2_ids', 'source_atomic_ids', 'sources', 'processing_type', 'merge_basis')
    placeholders = {'', '미지정', '없음', 'n/a', 'na', 'todo', 'null', 'none'}
    structure_rows, field_checks, text_checks = [], [], []
    normalized_names = defaultdict(list)
    for index, final in enumerate(predicted):
        gold_id = str(final.get('gold_id', '')).strip() or f'INDEX-{index}'
        missing = [key for key in required_fields if key not in final or final.get(key) in (None, '', [])]
        name, detail = str(final.get('requirement_name', '')).strip(), str(final.get('requirement_detail', '')).strip()
        normalized_names[_normalized_name(final)].append(gold_id)
        checks = {'name_length_ok': len(name) >= 4, 'detail_length_ok': len(detail) >= 20, 'name_not_placeholder': name.lower() not in placeholders, 'detail_not_placeholder': detail.lower() not in placeholders, 'detail_has_sentence_end': detail.endswith(('.', '다.', '함.', '한다.', '하여야 한다.', '해야 한다.'))}
        structure_rows.append({'gold_id': gold_id, 'missing_required_fields': '; '.join(missing), 'required_field_completeness': _round(1.0 - _ratio(len(missing), len(required_fields))), 'requirement_name_length': len(name), 'requirement_detail_length': len(detail), **checks, 'source_task2_count': len(dedupe_preserve(final.get('source_task2_ids', []))), 'source_atomic_count': len(dedupe_preserve(final.get('source_atomic_ids', []))), 'source_fur_count': len(dedupe_preserve(final.get('sources', [])))})
        field_checks.extend([not missing]); text_checks.extend(checks.values())
        if missing:
            review_rows.append({'severity': 'HIGH', 'category': 'STRUCTURE', 'item_id': gold_id, 'metric': 'missing_required_fields', 'value': '; '.join(missing), 'threshold': '0 fields', 'reason': '최종 GOLD 필수 필드가 비어 있음'})
        failed_text = [key for key, ok in checks.items() if not ok]
        if failed_text:
            review_rows.append({'severity': 'MEDIUM', 'category': 'TEXT_QUALITY', 'item_id': gold_id, 'metric': 'failed_text_checks', 'value': '; '.join(failed_text), 'threshold': 'all pass', 'reason': '요구사항명 또는 상세 문장의 기본 품질 점검 필요'})
    duplicate_names = {name: ids for name, ids in normalized_names.items() if name and len(ids) > 1}
    for name, ids in duplicate_names.items():
        review_rows.append({'severity': 'HIGH', 'category': 'DUPLICATE_NAME', 'item_id': '; '.join(ids), 'metric': 'normalized_name_duplicate', 'value': name, 'threshold': 'unique', 'reason': '공백을 제거한 요구사항명이 중복됨'})
    _save_csv(eval_dir / '02_GOLD_구조문장_평가.csv', structure_rows)

    # 3) TASK2·atomic·FUR 계보 완전성
    assignment = defaultdict(list)
    output_atomic_ids, output_fur_ids = set(), set()
    for final in predicted:
        gold_id = str(final.get('gold_id', '')).strip()
        for task2_id in dedupe_preserve(final.get('source_task2_ids', [])):
            assignment[task2_id].append(gold_id)
        output_atomic_ids.update(dedupe_preserve(final.get('source_atomic_ids', [])))
        output_fur_ids.update(dedupe_preserve(final.get('sources', [])))
    expected_task2_ids, expected_atomic_ids, expected_fur_ids = set(task2_by_id), set(atomic_by_id), set(fur_by_id)
    missing_task2, duplicate_task2 = sorted(expected_task2_ids - set(assignment)), {key: value for key, value in assignment.items() if len(set(value)) > 1}
    unknown_task2, missing_atomic, unknown_atomic = sorted(set(assignment) - expected_task2_ids), sorted(expected_atomic_ids - output_atomic_ids), sorted(output_atomic_ids - expected_atomic_ids)
    missing_fur, unknown_fur = sorted(expected_fur_ids - output_fur_ids), sorted(output_fur_ids - expected_fur_ids)
    lineage = {'expected_task2_count': len(expected_task2_ids), 'assigned_task2_count': len(set(assignment) & expected_task2_ids), 'missing_task2_ids': missing_task2, 'duplicate_task2_assignments': duplicate_task2, 'unknown_task2_ids': unknown_task2, 'expected_atomic_count': len(expected_atomic_ids), 'covered_atomic_count': len(expected_atomic_ids & output_atomic_ids), 'missing_atomic_ids': missing_atomic, 'unknown_atomic_ids': unknown_atomic, 'expected_fur_count': len(expected_fur_ids), 'covered_fur_count': len(expected_fur_ids & output_fur_ids), 'missing_fur_ids': missing_fur, 'unknown_fur_ids': unknown_fur, 'task2_coverage_rate': _round(_ratio(len(expected_task2_ids & set(assignment)), len(expected_task2_ids), 1.0)), 'atomic_coverage_rate': _round(_ratio(len(expected_atomic_ids & output_atomic_ids), len(expected_atomic_ids), 1.0)), 'fur_source_coverage_rate': _round(_ratio(len(expected_fur_ids & output_fur_ids), len(expected_fur_ids), 1.0))}
    for key, values in (('missing_task2_ids', missing_task2), ('unknown_task2_ids', unknown_task2), ('missing_atomic_ids', missing_atomic), ('unknown_atomic_ids', unknown_atomic), ('missing_fur_ids', missing_fur), ('unknown_fur_ids', unknown_fur)):
        if values:
            review_rows.append({'severity': 'HIGH', 'category': 'LINEAGE', 'item_id': '; '.join(values[:30]), 'metric': key, 'value': len(values), 'threshold': 0, 'reason': '입력→출력 계보 누락 또는 알 수 없는 ID 발생'})
    if duplicate_task2:
        review_rows.append({'severity': 'HIGH', 'category': 'LINEAGE', 'item_id': '; '.join(list(duplicate_task2)[:30]), 'metric': 'duplicate_task2_assignments', 'value': len(duplicate_task2), 'threshold': 0, 'reason': '하나의 TASK2가 여러 GOLD에 중복 배정됨'})

    # 4) 임베딩 의미 보존: TASK2→GOLD, 입력 FUR→GOLD, 요구사항명↔상세
    candidate_vectors = _encode_texts([_requirement_text(item) for item in candidates])
    final_vectors = _encode_texts([_requirement_text(item) for item in predicted])
    fur_vectors = _encode_texts([_fur_text(item) for item in furs])
    task2_index, final_index = {item['task2_id']: index for index, item in enumerate(candidates)}, {item['gold_id']: index for index, item in enumerate(predicted)}
    task2_final_sim = candidate_vectors @ final_vectors.T if len(candidates) and len(predicted) else np.zeros((len(candidates), len(predicted)))
    task2_rows, assigned_similarities = [], []
    for candidate in candidates:
        task2_id = candidate['task2_id']; gold_ids = assignment.get(task2_id, [])
        scores = [float(task2_final_sim[task2_index[task2_id], final_index[gold_id]]) for gold_id in gold_ids if gold_id in final_index]
        score = max(scores) if scores else 0.0
        assigned_similarities.append(score)
        row = {'task2_id': task2_id, 'assigned_gold_ids': '; '.join(gold_ids), 'action_type': candidate.get('action_type', ''), 'requirement_name': candidate.get('requirement_name', ''), 'semantic_similarity_to_assigned_gold': _round(score), 'passes_retention_threshold': score >= EVAL_TASK2_RETENTION_THRESHOLD}
        task2_rows.append(row)
        if score < EVAL_TASK2_RETENTION_THRESHOLD:
            review_rows.append({'severity': 'HIGH', 'category': 'SEMANTIC_RETENTION', 'item_id': task2_id, 'metric': 'task2_to_gold_similarity', 'value': _round(score), 'threshold': EVAL_TASK2_RETENTION_THRESHOLD, 'reason': 'TASK2 원문의 의미가 배정된 GOLD에 충분히 보존됐는지 검토 필요'})
    _save_csv(eval_dir / '03_TASK2_to_GOLD_의미보존.csv', task2_rows)

    fur_final_sim = fur_vectors @ final_vectors.T if len(furs) and len(predicted) else np.zeros((len(furs), len(predicted)))
    fur_rows, fur_lineage_scores = [], []
    for fur_no, fur in enumerate(furs):
        fur_id = str(fur['requirement_id']).strip()
        linked = [index for index, final in enumerate(predicted) if fur_id in dedupe_preserve(final.get('sources', []))]
        lineage_score = max((float(fur_final_sim[fur_no, index]) for index in linked), default=0.0)
        overall_index = int(np.argmax(fur_final_sim[fur_no])) if len(predicted) else -1
        overall_score = float(fur_final_sim[fur_no, overall_index]) if overall_index >= 0 else 0.0
        fur_lineage_scores.append(lineage_score)
        gold_ids = [predicted[index]['gold_id'] for index in linked]
        row = {'source_fur_id': fur_id, 'source_requirement_name': fur.get('requirement_name', ''), 'linked_gold_ids': '; '.join(gold_ids), 'linked_gold_count': len(gold_ids), 'lineage_semantic_max': _round(lineage_score), 'overall_semantic_max': _round(overall_score), 'overall_best_gold_id': predicted[overall_index]['gold_id'] if overall_index >= 0 else '', 'source_covered': bool(linked), 'passes_semantic_threshold': lineage_score >= EVAL_FUR_COVERAGE_THRESHOLD}
        fur_rows.append(row)
        if not linked or lineage_score < EVAL_FUR_COVERAGE_THRESHOLD:
            review_rows.append({'severity': 'HIGH' if not linked else 'MEDIUM', 'category': 'INPUT_COVERAGE', 'item_id': fur_id, 'metric': 'fur_to_gold_semantic_coverage', 'value': _round(lineage_score), 'threshold': EVAL_FUR_COVERAGE_THRESHOLD, 'reason': '입력 FUR가 출처 계보를 가진 GOLD에 의미상 충분히 반영됐는지 검토 필요'})
    _save_csv(eval_dir / '04_입력FUR_to_GOLD_커버리지.csv', fur_rows)

    name_vectors = _encode_texts([str(item.get('requirement_name', '')) for item in predicted])
    detail_vectors = _encode_texts([str(item.get('requirement_detail', '')) for item in predicted])
    name_detail_scores = [float(name_vectors[index] @ detail_vectors[index]) for index in range(len(predicted))] if len(predicted) else []
    for index, score in enumerate(name_detail_scores):
        structure_rows[index]['name_detail_semantic_similarity'] = _round(score)
        if score < EVAL_NAME_DETAIL_THRESHOLD:
            review_rows.append({'severity': 'MEDIUM', 'category': 'NAME_DETAIL_ALIGNMENT', 'item_id': predicted[index]['gold_id'], 'metric': 'name_detail_similarity', 'value': _round(score), 'threshold': EVAL_NAME_DETAIL_THRESHOLD, 'reason': '요구사항명과 상세 내용의 의미 정합성 검토 필요'})
    _save_csv(eval_dir / '02_GOLD_구조문장_평가.csv', structure_rows)

    # 5) 잔존 중복과 과병합 위험
    duplicate_rows: list[dict] = []
    strong_duplicate_rows: list[dict] = []
    review_duplicate_rows: list[dict] = []
    strong_duplicate_gold_ids: set[str] = set()

    if len(predicted) >= 2:
        duplicate_vectors = _encode_texts([_duplicate_evaluation_text(item) for item in predicted])
        duplicate_similarity = duplicate_vectors @ duplicate_vectors.T

        for left in range(len(predicted)):
            for right in range(left + 1, len(predicted)):
                left_item, right_item = predicted[left], predicted[right]
                cosine = float(duplicate_similarity[left, right])
                name_overlap = jaccard(
                    left_item.get('requirement_name', ''),
                    right_item.get('requirement_name', ''),
                )
                left_action = _normalize_action_type(left_item)
                right_action = _normalize_action_type(right_item)
                same_action = bool(left_action and right_action and left_action == right_action)

                is_strong_duplicate = same_action and (
                    (
                        cosine >= EVAL_RESIDUAL_STRONG_SIM
                        and name_overlap >= EVAL_RESIDUAL_STRONG_NAME_JACCARD
                    )
                    or (
                        cosine >= 0.985
                        and name_overlap >= 0.25
                    )
                )
                is_review_candidate = (
                    not is_strong_duplicate
                    and same_action
                    and cosine >= EVAL_RESIDUAL_REVIEW_SIM
                    and name_overlap >= EVAL_RESIDUAL_REVIEW_NAME_JACCARD
                )
                if not (is_strong_duplicate or is_review_candidate):
                    continue

                level = 'STRONG_DUPLICATE' if is_strong_duplicate else 'REVIEW_CANDIDATE'
                row = {
                    'candidate_level': level,
                    'left_gold_id': left_item['gold_id'],
                    'right_gold_id': right_item['gold_id'],
                    'left_name': left_item.get('requirement_name', ''),
                    'right_name': right_item.get('requirement_name', ''),
                    'left_action_type': left_item.get('action_type', ''),
                    'right_action_type': right_item.get('action_type', ''),
                    'same_action_type': same_action,
                    'semantic_similarity': _round(cosine),
                    'name_jaccard': _round(name_overlap),
                    'shared_source_furs': '; '.join(
                        sorted(set(left_item.get('sources', [])) & set(right_item.get('sources', [])))
                    ),
                }
                duplicate_rows.append(row)

                if is_strong_duplicate:
                    strong_duplicate_rows.append(row)
                    strong_duplicate_gold_ids.update((row['left_gold_id'], row['right_gold_id']))
                    review_rows.append({
                        'severity': 'HIGH',
                        'category': 'RESIDUAL_DUPLICATE',
                        'item_id': f"{row['left_gold_id']} | {row['right_gold_id']}",
                        'metric': 'strong_duplicate_pair',
                        'value': _round(cosine),
                        'threshold': (
                            f'sim>={EVAL_RESIDUAL_STRONG_SIM}, same_action=True, '
                            f'name_jaccard>={EVAL_RESIDUAL_STRONG_NAME_JACCARD}'
                        ),
                        'reason': '수행행위와 요구사항명 핵심어가 함께 일치하여 실제 잔존 중복 가능성이 높음',
                    })
                else:
                    review_duplicate_rows.append(row)
                    review_rows.append({
                        'severity': 'MEDIUM',
                        'category': 'RESIDUAL_DUPLICATE_REVIEW',
                        'item_id': f"{row['left_gold_id']} | {row['right_gold_id']}",
                        'metric': 'review_duplicate_pair',
                        'value': _round(cosine),
                        'threshold': (
                            f'sim>={EVAL_RESIDUAL_REVIEW_SIM}, same_action=True, '
                            f'name_jaccard>={EVAL_RESIDUAL_REVIEW_NAME_JACCARD}'
                        ),
                        'reason': '관련성이 높지만 자동 중복 확정 대상은 아니므로 필요할 때만 검토',
                    })

    _save_csv(
        eval_dir / '05_잔존중복_후보.csv',
        duplicate_rows,
        [
            'candidate_level',
            'left_gold_id',
            'right_gold_id',
            'left_name',
            'right_name',
            'left_action_type',
            'right_action_type',
            'same_action_type',
            'semantic_similarity',
            'name_jaccard',
            'shared_source_furs',
        ],
    )

    overmerge_rows = []
    for final in predicted:
        task2_ids = [task2_id for task2_id in dedupe_preserve(final.get('source_task2_ids', [])) if task2_id in task2_index]
        if len(task2_ids) < 2:
            continue
        indices = [task2_index[task2_id] for task2_id in task2_ids]
        pair_scores = [float(candidate_vectors[left] @ candidate_vectors[right]) for pos, left in enumerate(indices) for right in indices[pos + 1:]]
        final_no = final_index[final['gold_id']]
        retention_scores = [float(candidate_vectors[index] @ final_vectors[final_no]) for index in indices]
        row = {'gold_id': final['gold_id'], 'requirement_name': final.get('requirement_name', ''), 'source_task2_ids': '; '.join(task2_ids), 'source_task2_count': len(task2_ids), 'source_pair_min_similarity': _round(min(pair_scores)), 'source_pair_average_similarity': _round(_mean(pair_scores)), 'source_to_final_min_similarity': _round(min(retention_scores)), 'overmerge_review': min(pair_scores) < EVAL_OVERMERGE_PAIR_THRESHOLD or min(retention_scores) < EVAL_TASK2_RETENTION_THRESHOLD}
        overmerge_rows.append(row)
        if row['overmerge_review']:
            review_rows.append({'severity': 'MEDIUM', 'category': 'OVERMERGE_RISK', 'item_id': final['gold_id'], 'metric': 'source_pair_min_similarity', 'value': row['source_pair_min_similarity'], 'threshold': EVAL_OVERMERGE_PAIR_THRESHOLD, 'reason': '하나의 GOLD로 병합된 원본 TASK2 간 의미 거리가 커 과병합 가능성 검토 필요'})
    _save_csv(eval_dir / '06_과병합_검토대상.csv', overmerge_rows, ['gold_id', 'requirement_name', 'source_task2_ids', 'source_task2_count', 'source_pair_min_similarity', 'source_pair_average_similarity', 'source_to_final_min_similarity', 'overmerge_review'])

    # 6) 정답 GOLD가 있는 경우 exact + semantic 일대일 평가
    reference = None
    gold_path = find_gold_final_path(doc_id, input_path.parent) if SCORE_IF_GOLD_EXISTS else None
    reference_match_rows = []
    if gold_path is not None:
        gold_items = extract_gold_items(load_json(gold_path))
        predicted_names = {_normalized_name(item) for item in predicted if _normalized_name(item)}
        gold_names = {_normalized_name(item) for item in gold_items if _normalized_name(item)}
        exact_overlap = predicted_names & gold_names
        exact_precision = _ratio(len(exact_overlap), len(predicted_names)); exact_recall = _ratio(len(exact_overlap), len(gold_names)); exact_f1 = _ratio(2 * exact_precision * exact_recall, exact_precision + exact_recall)
        gold_vectors = _encode_texts([_requirement_text(item) for item in gold_items])
        matrix = final_vectors @ gold_vectors.T if len(predicted) and len(gold_items) else np.zeros((len(predicted), len(gold_items)))
        matches = _greedy_matches(matrix, EVAL_REFERENCE_MATCH_THRESHOLD)
        matched_pred, matched_gold = {row['predicted_index'] for row in matches}, {row['gold_index'] for row in matches}
        for row in matches:
            pred, gold = predicted[row['predicted_index']], gold_items[row['gold_index']]
            reference_match_rows.append({'predicted_gold_id': pred.get('gold_id', ''), 'predicted_name': pred.get('requirement_name', ''), 'reference_name': gold.get('requirement_name', gold.get('name', '')), 'semantic_similarity': row['semantic_similarity'], 'match_status': 'MATCHED'})
        for index, pred in enumerate(predicted):
            if index not in matched_pred:
                reference_match_rows.append({'predicted_gold_id': pred.get('gold_id', ''), 'predicted_name': pred.get('requirement_name', ''), 'reference_name': '', 'semantic_similarity': '', 'match_status': 'UNMATCHED_PREDICTION'})
        for index, gold in enumerate(gold_items):
            if index not in matched_gold:
                reference_match_rows.append({'predicted_gold_id': '', 'predicted_name': '', 'reference_name': gold.get('requirement_name', gold.get('name', '')), 'semantic_similarity': '', 'match_status': 'MISSED_REFERENCE'})
        sem_precision = _ratio(len(matches), len(predicted)); sem_recall = _ratio(len(matches), len(gold_items)); sem_f1 = _ratio(2 * sem_precision * sem_recall, sem_precision + sem_recall)
        count_similarity = 1.0 - _ratio(abs(len(predicted) - len(gold_items)), max(len(predicted), len(gold_items)), 1.0)
        reference_score = 100.0 * (0.75 * sem_f1 + 0.15 * exact_f1 + 0.10 * count_similarity)
        reference = {'gold_file': str(gold_path), 'predicted_count': len(predicted), 'gold_count': len(gold_items), 'count_difference': len(predicted) - len(gold_items), 'count_similarity': _round(count_similarity), 'exact_name_overlap': len(exact_overlap), 'exact_name_precision': _round(exact_precision), 'exact_name_recall': _round(exact_recall), 'exact_name_f1': _round(exact_f1), 'semantic_match_threshold': EVAL_REFERENCE_MATCH_THRESHOLD, 'semantic_match_count': len(matches), 'semantic_precision': _round(sem_precision), 'semantic_recall': _round(sem_recall), 'semantic_f1': _round(sem_f1), 'matched_average_similarity': _round(_mean([row['semantic_similarity'] for row in matches])), 'reference_score': _round(reference_score, 2)}
        _save_csv(eval_dir / '07_정답GOLD_일대일매칭.csv', reference_match_rows)
    else:
        print(f'[정답 기반 평가 건너뜀] {doc_id}: gold_final.json 없음', flush=True)

    # 7) 점수와 상태: 규칙 기반 내재 품질 점수와 정답 기반 점수는 분리
    field_rate = _ratio(sum(field_checks), len(field_checks), 1.0)
    text_rate = _ratio(sum(bool(value) for value in text_checks), len(text_checks), 1.0)
    task2_coverage = lineage['task2_coverage_rate']; atomic_coverage = lineage['atomic_coverage_rate']; fur_coverage = lineage['fur_source_coverage_rate']
    retention_average = _mean(assigned_similarities)
    retention_component = min(1.0, max(0.0, _ratio(retention_average - 0.55, 0.27)))
    # 약한 검토 후보는 점수를 깎지 않습니다.
    # 실제 강한 중복 후보에 포함된 GOLD 비율만 중복 청결도 점수에 반영합니다.
    duplicate_affected_rate = _ratio(len(strong_duplicate_gold_ids), max(len(predicted), 1))
    duplicate_component = max(0.0, 1.0 - duplicate_affected_rate)
    components = {'structure_completeness': _round(field_rate * 15, 2), 'task2_lineage': _round(task2_coverage * 20, 2), 'atomic_lineage': _round(atomic_coverage * 10, 2), 'fur_source_coverage': _round(fur_coverage * 10, 2), 'semantic_retention': _round(retention_component * 20, 2), 'residual_duplicate_cleanliness': _round(duplicate_component * 15, 2), 'text_quality': _round(text_rate * 10, 2)}
    intrinsic_score = round(sum(components.values()), 2)
    critical_count = sum((
        len(missing_task2),
        len(duplicate_task2),
        len(unknown_task2),
        len(missing_atomic),
        len(unknown_atomic),
        len(missing_fur),
        len(unknown_fur),
    )) + sum(1 for row in structure_rows if row['missing_required_fields'])
    high_review_count = sum(1 for row in review_rows if row['severity'] == 'HIGH')

    if critical_count > 0 or intrinsic_score < EVAL_REVIEW_SCORE:
        status = 'FAIL'
    elif (
        intrinsic_score < EVAL_PASS_SCORE
        or strong_duplicate_rows
        or high_review_count > 0
        or int(trace.get('task3_fallback_count', 0)) > 0
    ):
        status = 'REVIEW_REQUIRED'
    else:
        status = 'PASS'

    metrics = {
        'required_field_completeness_rate': _round(field_rate),
        'text_quality_check_pass_rate': _round(text_rate),
        'task2_to_gold_similarity_average': _round(retention_average),
        'task2_to_gold_similarity_minimum': _round(
            min(assigned_similarities) if assigned_similarities else 0.0
        ),
        'fur_to_gold_lineage_similarity_average': _round(_mean(fur_lineage_scores)),
        'fur_to_gold_lineage_similarity_minimum': _round(
            min(fur_lineage_scores) if fur_lineage_scores else 0.0
        ),
        'name_detail_similarity_average': _round(_mean(name_detail_scores)),
        'residual_strong_duplicate_pair_count': len(strong_duplicate_rows),
        'residual_duplicate_review_pair_count': len(review_duplicate_rows),
        'residual_duplicate_affected_gold_count': len(strong_duplicate_gold_ids),
        'overmerge_review_count': sum(1 for row in overmerge_rows if row['overmerge_review']),
        'review_item_count': len(review_rows),
        'high_review_item_count': high_review_count,
        'critical_error_count': critical_count,
    }

    report = {
        'evaluation_version': 'v10.5-residual-duplicate',
        'evaluated_at_utc': datetime.now(timezone.utc).isoformat(),
        'document_id': doc_id,
        'status': status,
        'intrinsic_quality_score': intrinsic_score,
        'score_components': components,
        'counts_and_transformation': counts,
        'lineage': lineage,
        'metrics': metrics,
        'thresholds': {
            'task2_retention': EVAL_TASK2_RETENTION_THRESHOLD,
            'fur_coverage': EVAL_FUR_COVERAGE_THRESHOLD,
            'name_detail': EVAL_NAME_DETAIL_THRESHOLD,
            'residual_strong_similarity': EVAL_RESIDUAL_STRONG_SIM,
            'residual_review_similarity': EVAL_RESIDUAL_REVIEW_SIM,
            'residual_strong_name_jaccard': EVAL_RESIDUAL_STRONG_NAME_JACCARD,
            'residual_review_name_jaccard': EVAL_RESIDUAL_REVIEW_NAME_JACCARD,
            'overmerge_pair': EVAL_OVERMERGE_PAIR_THRESHOLD,
            'reference_match': EVAL_REFERENCE_MATCH_THRESHOLD,
            'pass_score': EVAL_PASS_SCORE,
            'review_score': EVAL_REVIEW_SCORE,
        },
        'reference_gold_evaluation': reference,
        'review_items': review_rows,
        'interpretation': {
            'intrinsic_quality_score': (
                '정답 파일 없이 구조·계보·의미 보존·잔존 중복·문장 품질을 '
                '규칙과 임베딩으로 평가한 휴리스틱 점수'
            ),
            'reference_score': (
                '정답 GOLD가 있을 때 의미 기반 일대일 매칭과 이름 일치를 결합한 별도 점수'
            ),
            'residual_duplicate': (
                '강한 중복은 수행행위·명칭 핵심어·의미 유사도를 함께 만족한 쌍이며, '
                '검토 후보는 점수 감점 없이 참고 목록으로만 제공'
            ),
            'status': (
                'FAIL은 계보·필수필드 오류 또는 낮은 점수, REVIEW_REQUIRED는 '
                '강한 자동 검토 항목이나 fallback 존재, PASS는 자동 지표상 중대한 이상 없음'
            ),
        },
    }
    dump_json(eval_dir / '생성물_종합평가.json', report)
    dump_json(doc_out_dir / '생성물_종합평가.json', report)
    _save_csv(eval_dir / '08_검토필요_목록.csv', review_rows, ['severity', 'category', 'item_id', 'metric', 'value', 'threshold', 'reason'])
    summary_row = {'document_id': doc_id, 'evaluation_status': status, 'intrinsic_quality_score': intrinsic_score, **counts, **lineage, **metrics}
    summary_row = {key: value for key, value in summary_row.items() if not isinstance(value, (list, dict))}
    if reference:
        summary_row.update({f'reference_{key}': value for key, value in reference.items() if not isinstance(value, (list, dict)) and key != 'gold_file'})
    _save_csv(eval_dir / '00_생성물_평가요약.csv', [summary_row])
    print('=' * 72)
    print('[생성물 종합 평가]')
    print('document_id             :', doc_id)
    print('입력 FUR → T1 → T2 → GOLD:', len(furs), '→', len(atomics), '→', len(candidates), '→', len(predicted))
    print('내재 품질 점수          :', intrinsic_score)
    print('평가 상태               :', status)
    print('TASK2 의미 보존 평균    :', metrics['task2_to_gold_similarity_average'])
    print('입력 FUR 출처 커버리지  :', lineage['fur_source_coverage_rate'])
    print('강한 잔존 중복 후보     :', len(strong_duplicate_rows))
    print('잔존 중복 검토 후보     :', len(review_duplicate_rows))
    print('과병합 검토 대상        :', metrics['overmerge_review_count'])
    print('검토 필요 항목          :', len(review_rows))
    if reference:
        print('정답 의미 F1 / 점수     :', reference['semantic_f1'], '/', reference['reference_score'])
    print('평가 파일               :', eval_dir / '생성물_종합평가.json')
    print('=' * 72)
    return report


In [13]:
if RUN_USER_DOCUMENT_TEST:
    if RESET_OUTPUT_DIR_BEFORE_RUN and OUT_DIR.exists():
        print(f'[이전 출력 삭제] {OUT_DIR}', flush=True)
        shutil.rmtree(OUT_DIR)
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    input_files = discover_user_test_files()
    if len(input_files) != EXPECTED_TEST_DOCUMENT_COUNT:
        discovered = [str(path) for path in input_files]; all_json_files = [str(path) for path in sorted(USER_TEST_INPUT_DIR.glob('*.json'))]
        raise RuntimeError(f'실행 가능한 테스트 입력 문서 수가 {EXPECTED_TEST_DOCUMENT_COUNT}개가 아닙니다.\n- 발견한 입력 문서 수: {len(input_files)}\n- 입력 패턴: {USER_INPUT_GLOB}\n- 유효 입력 파일: {discovered}\n- 폴더 내 전체 JSON: {all_json_files}')
    print('=' * 72); print(f'[{EXPECTED_TEST_DOCUMENT_COUNT}개 테스트 문서 TASK1부터 신규 실행 + 종합 평가]')
    for index, input_path in enumerate(input_files, start=1): print(f'{index}. {input_path.name}')
    print('=' * 72)
    summary_rows, detail_results, evaluation_results, failed_documents, all_review_rows = [], {}, {}, [], []
    for document_no, input_path in enumerate(tqdm(input_files, desc='문서 TASK1~3 + 평가'), start=1):
        print('\n' + '=' * 72); print(f'[문서 {document_no}/{EXPECTED_TEST_DOCUMENT_COUNT} 시작] {input_path.name}'); print('=' * 72)
        try:
            final_requirements, relation_decisions, trace = run_pipeline_from_user_file(input_path)
            evaluation = evaluate_document_output(input_path, final_requirements, relation_decisions, trace)
            detail = {**trace, 'evaluation': evaluation}; detail_results[trace['document_id']] = detail; evaluation_results[trace['document_id']] = evaluation
            for review in evaluation.get('review_items', []): all_review_rows.append({'document_id': trace['document_id'], **review})
            reference = evaluation.get('reference_gold_evaluation') or {}
            row = {'status': 'SUCCESS', 'document_id': trace['document_id'], 'input_file': input_path.name, 'pipeline_quality_status': trace['quality_status'], 'evaluation_status': evaluation['status'], 'intrinsic_quality_score': evaluation['intrinsic_quality_score'], 'input_fur_count': trace['input_fur_count'], 'task1_atomic_total': trace['task1_atomic_total'], 'task2_candidate_total': trace['task2_candidate_total'], 'task3_final_total': trace['task3_final_total'], 'similar_component_count': trace['similar_component_count'], 'singleton_count': trace['singleton_count'], 'task3_fallback_count': trace['task3_fallback_count'], 'task2_coverage_rate': evaluation['lineage']['task2_coverage_rate'], 'atomic_coverage_rate': evaluation['lineage']['atomic_coverage_rate'], 'fur_source_coverage_rate': evaluation['lineage']['fur_source_coverage_rate'], 'task2_to_gold_similarity_average': evaluation['metrics']['task2_to_gold_similarity_average'], 'residual_strong_duplicate_pair_count': evaluation['metrics']['residual_strong_duplicate_pair_count'], 'residual_duplicate_review_pair_count': evaluation['metrics']['residual_duplicate_review_pair_count'], 'residual_duplicate_affected_gold_count': evaluation['metrics']['residual_duplicate_affected_gold_count'], 'overmerge_review_count': evaluation['metrics']['overmerge_review_count'], 'review_item_count': evaluation['metrics']['review_item_count'], 'reference_semantic_f1': reference.get('semantic_f1'), 'reference_score': reference.get('reference_score')}
            summary_rows.append(row); dump_json(OUT_DIR / f"{trace['document_id']}_실행상세.json", detail)
        except Exception as exc:
            failure = {'status': 'FAILED', 'input_file': str(input_path), 'error_type': type(exc).__name__, 'error_message': str(exc), 'traceback': traceback.format_exc()}
            failed_documents.append(failure); summary_rows.append({'status': 'FAILED', 'document_id': None, 'input_file': input_path.name, 'error_type': type(exc).__name__, 'error_message': str(exc)})
            dump_json(OUT_DIR / f'FAILED_{input_path.stem}.json', failure); print(f'[문서 실패] {input_path.name}: {type(exc).__name__}: {exc}', flush=True)
            if not CONTINUE_ON_DOCUMENT_ERROR:
                dump_json(OUT_DIR / '실패문서_목록.json', failed_documents); raise
        finally:
            gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()
    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(OUT_DIR / '3개문서_실행및평가요약.csv', index=False, encoding='utf-8-sig')
    dump_json(OUT_DIR / '3개문서_실행결과_전체.json', detail_results); dump_json(OUT_DIR / '3개문서_평가결과_전체.json', evaluation_results); dump_json(OUT_DIR / '실패문서_목록.json', failed_documents)
    _save_csv(OUT_DIR / '3개문서_검토필요_통합목록.csv', all_review_rows, ['document_id', 'severity', 'category', 'item_id', 'metric', 'value', 'threshold', 'reason'])
    success_count = sum(row.get('status') == 'SUCCESS' for row in summary_rows); pass_count = sum(row.get('evaluation_status') == 'PASS' for row in summary_rows); review_count = sum(row.get('evaluation_status') == 'REVIEW_REQUIRED' for row in summary_rows); eval_fail_count = sum(row.get('evaluation_status') == 'FAIL' for row in summary_rows)
    print('\n' + '=' * 72); print('[문서 실행 및 생성물 평가 종료]'); print('실행 성공/실패:', success_count, '/', len(failed_documents)); print('평가 PASS/REVIEW/FAIL:', pass_count, '/', review_count, '/', eval_fail_count); print('출력:', OUT_DIR); print('=' * 72)
    display(summary_df)
    if success_count != EXPECTED_TEST_DOCUMENT_COUNT: raise RuntimeError(f'테스트 문서 중 일부가 실패했습니다. 성공={success_count}, 실패={len(failed_documents)}')
else:
    print('[건너뜀] RUN_USER_DOCUMENT_TEST=False')


[이전 출력 삭제] /workspace/e2e_outputs_requirement_gold_v10_4
[3개 테스트 문서 TASK1부터 신규 실행 + 종합 평가]
1. DOC-001_기능전체_입력.json
2. DOC-003_기능전체_입력.json
3. DOC-021_기능전체_입력.json


문서 TASK1~3 + 평가:   0%|          | 0/3 [00:00<?, ?it/s]


[문서 1/3 시작] DOC-001_기능전체_입력.json
[JSON 저장 완료] /workspace/e2e_outputs_requirement_gold_v10_4/DOC-001/run_manifest.json (0.00MB)
[JSON 저장 완료] /workspace/e2e_outputs_requirement_gold_v10_4/DOC-001/input_user_document.json (0.03MB)


DOC-001 TASK1:   0%|          | 0/17 [00:00<?, ?it/s]

[생성 시작] TASK1_FUR_ATOMIC_DECOMPOSITION adapter=stage1 attempt=1/2 prompt=2,326 max_new=4,096 GPU_free=44.27GB
[JSON 저장 완료] /workspace/e2e_outputs_requirement_gold_v10_4/DOC-001/raw/task1/SFR-001_attempt1.json (0.00MB)
[JSON 저장 완료] /workspace/e2e_outputs_requirement_gold_v10_4/DOC-001/raw/task1/SFR-001.json (0.01MB)
[생성 완료] TASK1_FUR_ATOMIC_DECOMPOSITION generated=775 elapsed=0.78분 parse=strict
[JSON 저장 완료] /workspace/e2e_outputs_requirement_gold_v10_4/DOC-001/task1_atomics_by_fur.json (0.00MB)
[JSON 저장 완료] /workspace/e2e_outputs_requirement_gold_v10_4/DOC-001/run_manifest.json (0.00MB)
[생성 시작] TASK1_FUR_ATOMIC_DECOMPOSITION adapter=stage1 attempt=1/2 prompt=2,446 max_new=4,096 GPU_free=44.24GB
[JSON 저장 완료] /workspace/e2e_outputs_requirement_gold_v10_4/DOC-001/raw/task1/SFR-002_attempt1.json (0.01MB)
[JSON 저장 완료] /workspace/e2e_outputs_requirement_gold_v10_4/DOC-001/raw/task1/SFR-002.json (0.01MB)
[생성 완료] TASK1_FUR_ATOMIC_DECOMPOSITION generated=892 elapsed=0.88분 parse=strict
[JSON 저장 완

...

DOC-021 TASK2:   0%|          | 0/12 [00:00<?, ?it/s]

[생성 시작] TASK2_FUR_LOCAL_MERGE_NORMALIZE adapter=stage1 attempt=1/2 prompt=3,507 max_new=6,144 GPU_free=43.14GB
[JSON 저장 완료] /workspace/e2e_outputs_requirement_gold_v10_4/DOC-021/raw/task2/SFR-001_attempt1.json (0.01MB)
[JSON 저장 완료] /workspace/e2e_outputs_requirement_gold_v10_4/DOC-021/raw/task2/SFR-001.json (0.02MB)
[생성 완료] TASK2_FUR_LOCAL_MERGE_NORMALIZE generated=1,886 elapsed=2.07분 parse=strict
[JSON 저장 완료] /workspace/e2e_outputs_requirement_gold_v10_4/DOC-021/task2_by_fur/SFR-001.json (0.01MB)
[JSON 저장 완료] /workspace/e2e_outputs_requirement_gold_v10_4/DOC-021/run_manifest.json (0.00MB)
[생성 시작] TASK2_FUR_LOCAL_MERGE_NORMALIZE adapter=stage1 attempt=1/2 prompt=2,321 max_new=6,144 GPU_free=43.14GB
[JSON 저장 완료] /workspace/e2e_outputs_requirement_gold_v10_4/DOC-021/raw/task2/SFR-002_attempt1.json (0.00MB)
[JSON 저장 완료] /workspace/e2e_outputs_requirement_gold_v10_4/DOC-021/raw/task2/SFR-002.json (0.01MB)
[생성 완료] TASK2_FUR_LOCAL_MERGE_NORMALIZE generated=512 elapsed=0.51분 parse=strict
[JSO

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[JSON 저장 완료] /workspace/e2e_outputs_requirement_gold_v10_4/DOC-021/task3_embedding/00_similarity_distribution.json (0.00MB)
[JSON 저장 완료] /workspace/e2e_outputs_requirement_gold_v10_4/DOC-021/task3_embedding/01_similarity_edges.json (0.03MB)
[JSON 저장 완료] /workspace/e2e_outputs_requirement_gold_v10_4/DOC-021/task3_embedding/02_similarity_groups.json (0.01MB)
[임베딩 유사 후보 검색 v10]
전체 TASK2 후보     : 51
유사도 분포 q95/q97 : 0.911516 0.918594
적응형 유사도 하한  : 0.91
유효 유사 쌍        : 59
유사 그룹           : 10
유사 그룹 항목      : 41
독립 항목           : 10
최대 그룹 크기      : 8
[TASK3 컴포넌트] component=1, round=1, candidates=6, chunks=1
[생성 시작] TASK3_DOCUMENT_GLOBAL_DEDUP_FINALIZE adapter=stage3 attempt=1/2 prompt=2,737 max_new=4,105 GPU_free=43.14GB
[JSON 저장 완료] /workspace/e2e_outputs_requirement_gold_v10_4/DOC-021/raw/task3_embedding/component_001/round_01/group_001_attempt1.json (0.01MB)
[출력 구조 자동 보정] TASK3_DOCUMENT_GLOBAL_DEDUP_FINALIZE: relation_decisions[0].preserved_conditions:str->list; relation_decisions[1].pre

,status,document_id,input_file,pipeline_quality_status,evaluation_status,intrinsic_quality_score,input_fur_count,task1_atomic_total,task2_candidate_total,task3_final_total,...,atomic_coverage_rate,fur_source_coverage_rate,task2_to_gold_similarity_average,residual_strong_duplicate_pair_count,residual_duplicate_review_pair_count,residual_duplicate_affected_gold_count,overmerge_review_count,review_item_count,reference_semantic_f1,reference_score
0,SUCCESS,DOC-001,DOC-001_기능전체_입력.json,REVIEW_REQUIRED,FAIL,100.0,17,152,122,99,...,1.0,1.0,0.987973,0,3,0,0,4,0.842553,70.47
1,SUCCESS,DOC-003,DOC-003_기능전체_입력.json,PASS,PASS,100.0,10,27,27,19,...,1.0,1.0,0.978141,0,2,0,0,2,0.974359,84.12
2,SUCCESS,DOC-021,DOC-021_기능전체_입력.json,PASS,FAIL,100.0,12,64,51,25,...,1.0,1.0,0.962794,0,0,0,0,1,0.943396,79.68


## 실행 결과 및 평가 산출물

문서별 출력은 `/workspace/e2e_outputs_requirement_gold_v10_4/<DOC-ID>/`에 저장됩니다. 주요 생성물은 `예측_GOLD.json`, `예측_GOLD.csv`, `생성물_종합평가.json`과 `evaluation/` 폴더의 세부 CSV입니다.

평가 폴더에는 입력 대비 출력 변화, GOLD 구조·문장, TASK2→GOLD 의미 보존, 입력 FUR→GOLD 커버리지, 잔존 중복 후보, 과병합 검토 대상, 정답 GOLD 일대일 매칭, 검토 필요 목록이 저장됩니다.

`05_잔존중복_후보.csv`의 `candidate_level`은 다음과 같습니다.

- `STRONG_DUPLICATE`: 수행행위·명칭 핵심어·의미 유사도를 모두 충족한 강한 중복 후보이며 점수와 상태에 반영
- `REVIEW_CANDIDATE`: 관련성이 높아 참고할 후보지만 자동 감점하지 않음

`intrinsic_quality_score`는 정답 없이 계산하는 휴리스틱 점수이며, 정답 파일이 있으면 `reference_score`를 별도로 제공합니다.
